# Reproducing Chantranupong et al. (2023) from dandiset 001767

Paper. Chantranupong L, Beron CC, Zimmer JA, Wen MJ, Wang W, Sabatini BL (2023). Dopamine and glutamate regulate
striatal acetylcholine in decision-making. *Nature* 621, 577-585.
[doi:10.1038/s41586-023-06492-9](https://doi.org/10.1038/s41586-023-06492-9). The claims file
(`claims/001767/10.1038-s41586-023-06492-9.yaml`) was written from the version of record in Europe PMC (PMC10511323), with
the Methods, the Extended Data legends and the Supplementary Information; Supplementary Data 1 was not available.

Dandiset. [DANDI:001767](https://dandiarchive.org/dandiset/001767/0.260318.1448), version 0.260318.1448, the only
published version (the paper cites none; the draft has the same 379 files). It holds 379 NWB files, one per session, from
112 mice: demodulated fibre photometry at 54 ms per sample, its rolling z-score, binary port, lick, LED and laser channels,
and a trials table. Files are streamed; nothing is written to disk.

Outside input. The files carry no figure, cohort or genotype labels. The dandiset's metadata links its export code,
https://github.com/bernardosabatinilab/Chantranupong-2023-NWB, whose `manuscript_sessions.csv` (commit
f9a4ca95370e8d7451f2a29d81e184dd67a735ca, made before the version was published) lists each session with its figure,
genotype and virus codes. It is used for these labels only, and the link claim below checks it against the files. Method
details come from the paper's code, cited in the claims file: AchDA_manuscript_photometry at
cd8caf0e4a1b533e65ad3a2e3ec448516ee56871 and AchDA_manuscript_behavior at 20c0edb0d64466e2376f8a1f84ab97f055b13b44.

Claims. 47 claims from the paper and one link claim added here.

| Claim | Tested here | Why not |
|---|---|---|
| link-manuscript-sessions | yes | |
| ed1b-switch-after-block-transition, ed1hj-rflr-predicts-switching | yes, on the Fig. 1 mice | the n = 8 mice of Extended Data Fig. 1 are not identified |
| fig1e (three claims), ed3, fig1f, fig1g, fig1h (two claims), ed4bc | yes | |
| ed5-base-glm-r2 | run, blocked | its readings disagree (criterion) |
| ed5f-leave-out, ed5-history-glm-r2 | run, blocked | depend on ed5-base-glm-r2 |
| fig2h-photometry-glm, ed11-cko-glm-r2 | blocked, not run | depend on ed5-history-glm-r2 |
| fig2a, ed6c, fig2d, fig2e, fig2fg, ed6b | yes | |
| fig3d-telc-abolishes-ach | blocked | its Ach3.0 TelC and mCherry mice are not in the dataset |
| fig3c, fig4ef (slice recordings), fig5a (histology), fig3g (DAN inhibition), fig4bc (head-fixed) | blocked | not in the dataset |
| fig3e, fig3f, ed8g, ed9jm, ed7e | yes | |
| ed8f-amplitude-no-consistent-effect | run, blocked | the paper does not say which DA sensor, and the two give different outcomes |
| fig4g (two claims), ed11e, ed11fh, ed11i, ed11j | yes | |
| fig5b, fig5c, fig5d, fig5ef-input-specific-ach, ed13 | yes | |
| fig5ef-input-telc-dampens-ach | run in part, blocked | the thalamic mCherry mice have no raw traces |
| ed2op-mutant-sensors-flat | yes (Ach3.0 mutant only) | no dLight1.1 mutant in the dataset |

Windows W1 (post = samples +1 to +17 after side entry, pre = -17 to -1) and W2 (post = 0 to +16, pre = -16 to 0), the
per-trial measures, the per-mouse test T1 (two-sample t-test with pooled variance) and the test across mice T2 are the claims
file's header. Every result below is computed from the streamed data.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import achda

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
assets = achda.list_assets()
S = achda.load_all(assets, workers=32)
print(f"loaded {len(S)} files from dandiset {achda.DANDISET_ID} version {achda.DANDISET_VERSION} in {time.time() - started:.0f} s")
meta = achda.dandiset_metadata()
print("archive summary:", meta["assetsSummary"]["numberOfFiles"], "files,", meta["assetsSummary"]["numberOfSubjects"], "subjects;",
      "related resources:", [r["url"] for r in meta.get("relatedResource", [])])

## Data check

Each check of the metadata against the values is stated under its output. Cohorts come from the export table
`manuscript_sessions.csv` (see the header); everything else from the files.

In [ ]:
csv = achda.manuscript_sessions()
key = {(s["mouse"], s["date"]): s for s in S}
csv["in_dataset"] = [(m, d) in key for m, d in zip(csv.Mouse, csv.date)]
uses = csv[csv.in_dataset]
def figmice(fig, genotype=None):
    u = uses[uses.Figure == fig]
    if genotype is not None:
        u = u[u.Genotype == genotype]
    return sorted(u.Mouse.unique())
def sessions_of(mice, fig=None):
    u = uses if fig is None else uses[uses.Figure == fig]
    keys = set(zip(u.Mouse, u.date))
    return [s for s in S if s["mouse"] in mice and (s["mouse"], s["date"]) in keys]
print("sessions in the dataset:", len(S), "; CSV rows:", len(csv), "; unique CSV sessions:", csv[["Mouse","date"]].drop_duplicates().shape[0])
print("dataset sessions found in the CSV:", sum((s["mouse"], s["date"]) in set(zip(csv.Mouse, csv.date)) for s in S))
print(csv.groupby(["Figure", "Genotype"]).agg(rows=("in_dataset", "size"), in_dataset=("in_dataset", "sum"), mice=("Mouse", "nunique")).to_string())

Every one of the 379 files is in the export table, which lists 443 sessions: 64 sessions and 21 mice it names are not
in the dataset (printed below). The table's figure labels do not follow the published numbering (EDFig10, EDFig12 and EDFig16
where the paper has 13 Extended Data figures); from their mice and sensors, EDFig10 is the GLM cohort of Extended Data Fig. 5,
EDFig12 the CIN-stimulation cohort of Extended Data Fig. 7 and EDFig16 the rDAh cohort of Extended Data Fig. 8.

In [ ]:
import hashlib
# mice the CSV names that the dataset lacks, by figure
miss = csv[~csv.in_dataset].groupby(["Figure", "Genotype"]).apply(lambda g: sorted(set(g.Mouse) - set(uses.Mouse)), include_groups=False)
print("Mice named in the CSV with no session in the dataset:"); print(miss[miss.map(len) > 0].to_string())
part = csv[~csv.in_dataset].groupby(["Figure", "Genotype"]).apply(lambda g: sorted(set(g.Mouse) & set(uses.Mouse)), include_groups=False)
print("\nMice with some CSV sessions missing:"); print(part[part.map(len) > 0].to_string())
# sampling
rates = pd.Series([round(s.get("rate", np.nan), 4) for s in S]); steps = pd.Series([s["params"]["finalTimeStep"] for s in S])
print("\nSampling rate (Hz) of the photometry series:", rates.value_counts().to_dict(), "; finalTimeStep (s):", steps.value_counts().to_dict())
lens_ok = all(len(v) == s["n_samples"] for s in S for v in s["z"].values())
print("every trace has as many samples as the event channels:", lens_ok)
# sensors
rows = []
for s in S:
    for ch, m in s["meta"].items():
        if ch in s["z"]:
            rows.append(dict(mouse=s["mouse"], date=s["date"], ch=ch, sensor=m["sensor"], region=m["brainRegion"], hemi=m["hemisphere"], virus=m["virus"].split(" ")[0],
                             zmean=float(np.mean(s["z"][ch])), zsd=float(np.std(s["z"][ch])), has_raw=ch in s["raw"], has_f0=ch in s["f0"],
                             rawmin=float(s["raw"][ch].min()) if ch in s["raw"] else np.nan, rawmax=float(s["raw"][ch].max()) if ch in s["raw"] else np.nan,
                             f0_over_rollmean=float(np.median(s["f0"][ch]) / np.median(s["raw"][ch])) if ch in s["raw"] and ch in s["f0"] else np.nan, nan=int(np.isnan(s["z"][ch]).sum())))
chan = pd.DataFrame(rows)
st = pd.DataFrame([dict(mouse=s["mouse"], ch=ch, sensor=s["meta"][ch]["sensor"], **h) for s in S for ch, h in s["streams"].items()])
st["named"] = st.sensor != "unknown"
print("\nStreams present per channel, by whether the notes name a sensor (channel-sessions):")
print(st.groupby(["named", "raw", "z", "f0"]).size().to_string())
lost = st[st.named & ~st.z]
print("channels with a named sensor and no z-scored trace:", len(lost), "in", lost.mouse.nunique(), "mice:", sorted(lost.mouse.unique()))
lostraw = st[st.named & st.z & ~st.raw]
print("channels with a z-scored trace but no raw trace:", len(lostraw), "in", lostraw.mouse.nunique(), "mice:", sorted(lostraw.mouse.unique()))
print("\nChannels by sensor (sessions):"); print(chan.groupby(["sensor", "region"]).size().to_string())
print("\nz-scored traces: mean of session means %.3f (range %.3f to %.3f), s.d. range %.3f to %.3f, NaN samples %d" % (chan.zmean.mean(), chan.zmean.min(), chan.zmean.max(), chan.zsd.min(), chan.zsd.max(), chan.nan.sum()))
print("raw demodulated traces (a.u.): min %.3g, max %.3g" % (chan.rawmin.min(), chan.rawmax.max()))
print("stored baseline_f0 / median raw: median %.4f, range %.4f to %.4f" % (chan.f0_over_rollmean.median(), chan.f0_over_rollmean.min(), chan.f0_over_rollmean.max()))
# virus codes in the files against the CSV
vmap = {"greenL": "Ch5_Virus", "redL": "Ch6_Virus", "greenR": "Ch1_Virus", "redR": "Ch2_Virus"}
agree = []
for _, r in chan.iterrows():
    c = uses[(uses.Mouse == r.mouse) & (uses.date == r.date)]
    v = set(c[vmap[r.ch]].dropna())
    agree.append(r.virus in v or (r.virus.replace("B", ".B") in v) if v else None)
chan["virus_agrees_csv"] = agree
print("\nchannel virus code equals the CSV's:", pd.Series(agree).value_counts(dropna=False).to_dict())
print(chan[chan.virus_agrees_csv == False][["mouse", "date", "ch", "sensor", "virus"]].drop_duplicates(["mouse","ch"]).to_string())
# duplicates
h = {}
for s in S:
    for ch, v in s["z"].items():
        h.setdefault(hashlib.md5(np.round(v, 4).tobytes()).hexdigest(), []).append(f'{s["mouse"]}_{s["date"]}_{ch}')
dups = [v for v in h.values() if len(v) > 1]
print("\nidentical z-scored traces deposited more than once:", dups if dups else "none")
hb = {}
for s in S:
    t = s["trials"]; hb.setdefault(hashlib.md5(t[["chose_left", "was_rewarded"]].values.tobytes() + t.center_in_time.round(3).values.tobytes()).hexdigest(), []).append(f'{s["mouse"]}_{s["date"]}')
print("identical trial tables:", [v for v in hb.values() if len(v) > 1] or "none")
# trials
tt = pd.DataFrame([dict(mouse=s["mouse"], n=len(s["trials"]), photo=int(s["trials"].has_all_photometry_data.sum()), rew=float(s["trials"].was_rewarded.mean()),
                        both=int((s["trials"].chose_left & s["trials"].chose_right).sum()), neither=int((~s["trials"].chose_left & ~s["trials"].chose_right).sum()),
                        probs=tuple(sorted(set(np.round(s["trials"].left_reward_prob, 2)))), side_laser=int(s["trials"].side_laser.sum()), center_laser=int(s["trials"].center_laser.sum()),
                        laser_onsets=len(s["events"]["laser_or_unused"]), led=len(s["events"]["side_LED"]), licks=len(s["events"]["left_lick"]) + len(s["events"]["right_lick"])) for s in S])
print("\ntrials per session: %d to %d (median %d); with full photometry window: median %d; reward rate %.2f to %.2f" % (tt.n.min(), tt.n.max(), tt.n.median(), tt.photo.median(), tt.rew.min(), tt.rew.max()))
print("trials choosing both ports:", tt.both.sum(), "; neither:", tt.neither.sum())
print("left reward probabilities:", tt.probs.value_counts().to_dict())
print("sessions with laser trials flagged in the trials table:", int((tt.side_laser + tt.center_laser > 0).sum()), "; sessions with laser-channel onsets:", int((tt.laser_onsets > 0).sum()), sorted(set(tt[tt.laser_onsets > 0].mouse)))
print("sessions without LED onsets:", int((tt.led == 0).sum()), "; without lick onsets:", int((tt.licks == 0).sum()))
print("mice without LED or lick channels:", sorted(set(tt[(tt.led == 0) & (tt.licks == 0)].mouse)))

print("Subject metadata in the files (age, sex, genotype), sessions:", pd.Series([(s["subject"].get("age"), s["subject"].get("sex"), s["subject"].get("genotype", "none")) for s in S]).value_counts().to_dict())
co = pd.Series([float(((s["trials"].photometry_center_out_index > s["trials"].photometry_side_in_index) & (s["trials"].photometry_side_in_index >= 0)).mean()) for s in S])
print("fraction of trials whose centre-exit sample is after the side-entry sample: median %.3f, max %.3f, sessions above 0.1: %d" % (co.median(), co.max(), int((co > 0.1).sum())))

What the checks show:

- All series are at 18.52 Hz (54 ms per sample), as the files' processing parameters and the Methods state; traces and event
  channels have the same length in every file.
- The z-scored traces have mean near 0 and s.d. near 1 in every session, consistent with the 60-s rolling z-score their
  description gives (a rolling z-score of the raw trace over 1111 samples reproduces them, r = 0.9996, checked while
  preparing on one session; not run here).
- Many channels lack streams: channels whose notes name a sensor have no raw trace in 193 channel-sessions and no z-scored
  trace in 30 (listed above). This removes the right-hemisphere dLight1.1 of the CIN-stimulation mice, the raw traces of the
  thalamic mCherry mice, and some Ach channels of the Fig. 5 and Drd2 cohorts.
- The stored `baseline_f0` is about 0.07 times the raw trace (median ratio above), so it is not the F0 of the raw trace;
  dF/F0 is computed here from the raw trace (helper docstring).
- The `sensor` field holds virus codes in S950, S952, S1142 and S1143 ('279B', '333 + 304'), and names three viruses only by
  number ('unknown (virus 216)', 'virus 390', and 222 inside a compound code).
- No recording is deposited twice: no two z-scored traces and no two trial tables are identical.
- The trials tables' laser columns are False in every session, while the laser channel has onsets in the seven
  CIN-stimulation sessions.
- Subject metadata is a placeholder in every file (age P0D, sex U, no genotype); the paper used male mice 6 to 8 weeks old,
  and genotype is needed for Figs. 3 to 5.
- The high-port reward probability is 0.95 in 180 of 379 sessions and 0.8 to 1.0 in the others.
- In a median of 17% of trials per session (up to 43%) the trials table's centre-exit sample comes after its side-entry
  sample, which cannot happen in the task; centre exits are taken here from the port's occupancy channel, and decision time
  from centre entry to side entry.

In [ ]:
# 1. trials-table times against the photometry indices of the same trials
off = []
for s in S:
    t = s["trials"]; ok = t.has_all_photometry_data & (t.photometry_side_in_index >= 0) & (t.photometry_center_in_index >= 0)
    if ok.sum() < 5:
        continue
    d_side = t.side_in_time[ok] / achda.DT - t.photometry_side_in_index[ok]
    d_cent = t.center_in_time[ok] / achda.DT - t.photometry_center_in_index[ok]
    # side-entry index against the port event channel (choice port onset at the index)
    ev = np.r_[s["events"]["left_port"], s["events"]["right_port"]]
    hit = np.isin(t.photometry_side_in_index[ok].values, ev).mean()
    cev = np.isin(t.photometry_center_in_index[ok].values, s["events"]["center_port"]).mean()
    off.append(dict(session=f'{s["mouse"]}_{s["date"]}', offset_s=float(np.median(d_side)) * achda.DT, offset_sd_samples=float(np.std(d_side)),
                    centre_minus_side_offset=float(np.median(d_cent) - np.median(d_side)), side_index_on_port_onset=hit, centre_index_on_port_onset=cev))
off = pd.DataFrame(off)
print("Offset of trials-table times from the photometry clock (side_in_time/0.054 s minus the side-entry sample), per session:")
print(off[["offset_s", "offset_sd_samples", "centre_minus_side_offset", "side_index_on_port_onset", "centre_index_on_port_onset"]].describe().round(3).to_string())
# 2. contrast: rewarded minus unrewarded mean z-scored DA at each lag about side entry, against shuffled reward labels
rng = np.random.default_rng(0)
lags = np.arange(-30, 31)
prof, null = [], []
for s in S:
    chans = achda.channel_of(s, achda.DA_SENSORS)
    if not chans:
        continue
    t = achda.photometry_trials(s); z = s["z"][chans[0]]
    if t.was_rewarded.sum() < 10 or (~t.was_rewarded).sum() < 10:
        continue
    seg, keep = achda.segments(z, t.photometry_side_in_index.values[:, None] + lags[None, :])
    r = t.was_rewarded.values[keep]
    prof.append(seg[r].mean(0) - seg[~r].mean(0))
    rp = rng.permutation(r); null.append(seg[rp].mean(0) - seg[~rp].mean(0))
prof, null = np.array(prof), np.array(null)
pk = lags[np.abs(prof).argmax(1)]
print(f"\nDA sessions: {len(prof)}. Lag (samples) of the largest |rewarded - unrewarded| difference: median {np.median(pk):.0f}, "
      f"{(pk > 0).mean():.2f} of sessions after side entry; mean |difference| before entry (lags -30..-1) {np.abs(prof[:, lags < 0]).mean():.3f}, "
      f"after (+1..+30) {np.abs(prof[:, lags > 0]).mean():.3f}; same with shuffled labels {np.abs(null[:, lags < 0]).mean():.3f} and {np.abs(null[:, lags > 0]).mean():.3f}")

The trials tables' `photometry_*_index` columns land on the port onsets of the event channels in every session
(fraction 1.0), so the photometry and the behaviour events share one clock. The tables' times (`start_time`,
`side_in_time`) do not: they are offset from the photometry series by a median of -4.6 s (range -125 to +1.2 s) and in one
session drift within it. All analyses here use the sample indices. The contrast check: the rewarded minus unrewarded DA
difference peaks after side entry in 97% of the DA sessions and is about six times its shuffled-label value after entry,
against 1.4 times before it, so the trial table's outcomes and side entries are aligned with the photometry.

## link-manuscript-sessions

> The sessions table of the dandiset's export code (`manuscript_sessions.csv`) can be matched to the dandiset's files by
> mouse and session date, and its virus codes agree with the files' channel metadata.

In [ ]:
# link-manuscript-sessions: the export table's sessions against the files, and its virus codes against the channel metadata
ds_keys = {(s["mouse"], s["date"]) for s in S}
csv_keys = set(zip(csv.Mouse, csv.date))
print(f"dataset sessions matched by mouse and date: {len(ds_keys & csv_keys)} of {len(ds_keys)}; CSV sessions not in the dataset: {len(csv_keys - ds_keys)} of {len(csv_keys)}")
print(f"channel virus codes equal to the CSV's: {int(chan.virus_agrees_csv.sum())} of {chan.virus_agrees_csv.notna().sum()} channel-sessions "
      f"({chan.virus_agrees_csv.mean():.3f})")
print("CSV genotype per mouse is unique:", bool(uses.groupby("Mouse").Genotype.nunique().max() == 1),
      "; mice with more than one genotype label:", uses.groupby("Mouse").Genotype.nunique().loc[lambda x: x > 1].index.tolist())
print("cohort sizes by the CSV against the paper: Drd2 WT %d (12), f/f %d (7), cKO %d (8); Fig2 %d (6); striatum-wide TelC/mCherry %d/%d (5/5)" % (
    len(figmice("Fig4", "Chat jax") + figmice("Fig4", "Chat Jax")), len(figmice("Fig4", "Chat Jax neg D2R f/f")), len(figmice("Fig4", "Chat Jax D2R KO")), len(figmice("Fig2")),
    len([m for m in figmice("Fig3") if any("SIO-tetanus" in s["meta"][c]["sensor"] for s in S if s["mouse"] == m for c in s["meta"])]),
    len([m for m in figmice("Fig3") if m in ("S1233", "S1234", "S1246", "S1248", "S1260")])))

**Result: reproduced.** All 379 files match a row of the table by mouse and date, 97% of channel virus codes equal the
table's (the rest are compound codes written in one place and not the other, and one session of WT65), each mouse has one
genotype label, and the table's cohorts have the paper's sizes where the paper gives them. The claims below that select
mice by figure or genotype rest on this link.

## Fig. 1 trials

The Fig. 1 cohort is the sessions the export table labels Fig1. The DA channel follows aa_processStats_final_Fig1_AllDA.m
(channel 1, greenR; channel 5, greenL, in WT8 to WT11, WT21 and WT22), kept when it is dLight1.1 in the VLS; the Ach channel
is the Ach3.0 channel (greenL). Each trial with a full photometry window gives one row per window reading.

In [ ]:
# Fig. 1 cohort: sessions the CSV labels Fig1. DA channel as in aa_processStats_final_Fig1_AllDA.m (channel 1 = greenR,
# channel 5 = greenL in WT8-WT11, WT21, WT22), kept when its sensor is dLight1.1 in the VLS; Ach channel: the Ach3.0 channel
# (greenL = channel 5, aa_processStats_final_Fig1_Ach_Words.m).
SWAPPED_DA = {"WT8", "WT9", "WT10", "WT11", "WT21", "WT22"}
def fig1_channel(s, kind):
    if kind == "DA":
        ch = "greenL" if s["mouse"] in SWAPPED_DA else "greenR"
        ok = ch in s["z"] and s["meta"][ch]["sensor"] == "dlight1.1" and s["meta"][ch]["brainRegion"] == "VLS"
        return ch if ok else None
    chans = achda.channel_of(s, achda.ACH_SENSORS, prefer="greenL")
    return chans[0] if chans else None

def trial_measures(sessions, chan_fn, readings=("W1", "W2", "CODE"), extra=None):
    """One row per trial and window reading: mean, range (max-min), min and max of the z-scored trace, pre and post."""
    rows = []
    for s in sessions:
        ch = chan_fn(s)
        if ch is None:
            continue
        t = achda.photometry_trials(s); z = s["z"][ch]
        side = t.photometry_side_in_index.values
        for rd in readings:
            for which in ("pre", "post"):
                seg, keep = achda.segments(z, achda.window_idx(side, which, rd))
                tk = t[keep]
                d = pd.DataFrame({"mouse": s["mouse"], "session": s["date"], "chan": ch, "hemi": s["meta"][ch]["hemisphere"], "reading": rd, "which": which,
                                  "trial": tk.trial.values, "rew": tk.was_rewarded.values, "left": tk.chose_left.values, "word": tk.word.values,
                                  "switch": tk.switch.values, "mean": seg.mean(1), "range": seg.max(1) - seg.min(1), "min": seg.min(1), "max": seg.max(1)})
                rows.append(d)
    return pd.concat(rows, ignore_index=True)

f1 = sessions_of(figmice("Fig1"), "Fig1")
M = {k: trial_measures(f1, lambda s, k=k: fig1_channel(s, k)) for k in ("DA", "Ach")}
for k in M:
    m = M[k][(M[k].reading == "W1") & (M[k].which == "post")]
    print(f"{k}: {m.mouse.nunique()} mice, {m.groupby('mouse').session.nunique().sum()} sessions, {len(m)} trials; mice: {sorted(m.mouse.unique(), key=lambda x: int(x[2:]))}")

def per_mouse(df, cond_a, cond_b, measure, label=""):
    """T1 per mouse (cond_a minus cond_b) and T2 across mice."""
    out = []
    for mouse, g in df.groupby("mouse"):
        a = g.loc[cond_a(g), measure].values; b = g.loc[cond_b(g), measure].values
        eff, p = achda.t1(a, b)
        out.append(dict(mouse=mouse, n_a=len(a), n_b=len(b), mean_a=a.mean() if len(a) else np.nan, mean_b=b.mean() if len(b) else np.nan, diff=eff, p=p))
    out = pd.DataFrame(out)
    ok = out.dropna(subset=["mean_a", "mean_b"])
    t2 = stats.ttest_rel(ok.mean_a, ok.mean_b).pvalue if len(ok) > 1 else np.nan
    return out, t2

def summarise(out, t2, direction):
    sig = (out.p < 0.05)
    dirn = np.sign(out["diff"]) == direction
    return dict(mice=len(out), sig_in_direction=int((sig & dirn).sum()), sig_other=int((sig & ~dirn).sum()), median_diff=float(out["diff"].median()), T2_p=float(t2))

The cohort has 14 DA mice and 16 Ach mice, where the paper has 13 and 14; the paper's scripts read their mice from data
folders, so which ones it left out cannot be told. All are kept.

## fig1e-reward-outcome-post and fig1e-reward-outcome-not-pre

> After side-port entry, unrewarded trials show a lower mean DA and a larger delta-Ach than rewarded trials, in DA (13 mice)
> and Ach (14 mice) recordings.

> Before side-port entry, mean DA and delta-Ach do not differ significantly between rewarded and unrewarded trials.

In [ ]:
# fig1e-reward-outcome-post and fig1e-reward-outcome-not-pre
REW, NOREW = (lambda g: g.rew), (lambda g: ~g.rew)
res = []
for k, meas, direction in (("DA", "mean", -1), ("Ach", "range", +1)):
    for rd in ("W1", "W2", "CODE"):
        for which in ("post", "pre"):
            d = M[k][(M[k].reading == rd) & (M[k].which == which)]
            out, t2 = per_mouse(d, NOREW, REW, meas)
            res.append(dict(signal=k, measure=meas, reading=rd, window=which, **summarise(out, t2, direction), any_sig=int((out.p < 0.05).sum())))
fig1e = pd.DataFrame(res)
print("Unrewarded minus rewarded, T1 per mouse; direction = lower DA mean / higher delta-Ach in unrewarded trials:")
print(fig1e.round(4).to_string(index=False))
d = M["DA"][(M["DA"].reading == "W1") & (M["DA"].which == "post")]; out, _ = per_mouse(d, NOREW, REW, "mean")
print("\nPer mouse, DA post W1:"); print(out.round(4).to_string(index=False))
d = M["Ach"][(M["Ach"].reading == "W1") & (M["Ach"].which == "post")]; out, _ = per_mouse(d, NOREW, REW, "range")
print("\nPer mouse, delta-Ach post W1:"); print(out.round(4).to_string(index=False))

**Result: reproduced (fig1e-reward-outcome-post).** After side entry mean DA is lower in unrewarded trials in 14 of 14 mice
with T1 p < 0.05 under W1, W2 and the code's window (median difference -0.80 z under W1), and delta-Ach is higher in 12 of 16
mice under each window (median +0.62 z under W1); 2 mice (W1) and 1 (W2) show a significant difference the other way. T2
across mice gives p < 0.001 for both signals.

**Result: reproduced, resting on an assumption (fig1e-reward-outcome-not-pre).** Before side entry 6 of 14 DA mice differ
significantly under W1 and under the code's window, all with lower DA in unrewarded trials (median -0.05 z, T2 p = 0.0045),
and 7 of 14 under W2, which is not fewer than half. For Ach 3 of 16 mice differ under W1 and W2, all in the direction
opposite to the post effect. The outcome takes W1, in which "before side entry" excludes the side-entry sample (the paper's
Fig. 1f, Fig. 3 and Drd2 scripts drop it with startDrop = -1); W2 puts the DA count at exactly half.

## fig1e-lda-accuracy

> An LDA classifier separates rewarded from unrewarded trials with accuracy above 80% when trained on post-side-entry signals
> and around 50% (chance) when trained on pre-side-entry signals.

In [ ]:
# fig1e-lda-accuracy: per mouse, each trial's window trace as features, rewarded against unrewarded
rng = np.random.default_rng(107)
rows = []
for k in ("DA", "Ach"):
    ch_fn = lambda s, k=k: fig1_channel(s, k)
    for mouse in sorted(M[k].mouse.unique()):
        ss = [s for s in f1 if s["mouse"] == mouse and ch_fn(s)]
        for rd in ("W1", "W2"):
            for which in ("post", "pre"):
                X, y = [], []
                for s in ss:
                    t = achda.photometry_trials(s)
                    seg, keep = achda.segments(s["z"][ch_fn(s)], achda.window_idx(t.photometry_side_in_index.values, which, rd))
                    X.append(seg); y.append(t.was_rewarded.values[keep])
                X, y = np.vstack(X), np.concatenate(y)
                rows.append(dict(signal=k, mouse=mouse, reading=rd, window=which, n_rew=int(y.sum()), n_norew=int((~y).sum()),
                                 acc=achda.lda_accuracy(X[y], X[~y], rng)))
lda = pd.DataFrame(rows)
summ = lda.groupby(["signal", "reading", "window"]).acc.agg(["mean", "min", "max", "count"]).reset_index()
summ["all_above_80"] = lda.groupby(["signal", "reading", "window"]).acc.apply(lambda a: bool((a > 0.8).all())).values
print("LDA test accuracy (mean of 10 balanced 70/30 splits per mouse), summarised over mice:")
print(summ.round(3).to_string(index=False))
print("\nPer mouse (W1):"); print(lda[lda.reading == "W1"].pivot_table(index=["signal", "mouse"], columns="window", values="acc").round(3).to_string())
# shuffled-label reference
rng2 = np.random.default_rng(1071); ref = []
for k in ("DA", "Ach"):
    for mouse in sorted(M[k].mouse.unique())[:4]:
        d = M[k][(M[k].mouse == mouse) & (M[k].reading == "W1") & (M[k].which == "post")]
        ss = [s for s in f1 if s["mouse"] == mouse and fig1_channel(s, k)]
        X = np.vstack([achda.segments(s["z"][fig1_channel(s, k)], achda.window_idx(achda.photometry_trials(s).photometry_side_in_index.values, "post", "W1"))[0] for s in ss])
        y = rng2.permutation(np.concatenate([achda.photometry_trials(s).was_rewarded.values[achda.segments(s["z"][fig1_channel(s, k)], achda.window_idx(achda.photometry_trials(s).photometry_side_in_index.values, "post", "W1"))[1]] for s in ss]))
        ref.append(dict(signal=k, mouse=mouse, shuffled_acc=achda.lda_accuracy(X[y], X[~y], rng2)))
print("\nShuffled-label reference (post W1, first 4 mice per signal):"); print(pd.DataFrame(ref).round(3).to_string(index=False))

**Result: reproduced.** Under reading (a), the mean over mice, post-entry accuracy is 0.934 (W1) and 0.935 (W2) for DA and
0.837 and 0.840 for Ach, above 0.80, and pre-entry accuracy is 0.516 and 0.518 for DA and 0.539 and 0.542 for Ach, within
50 ± 5%. Under reading (b), every mouse, DA post accuracy exceeds 0.80 in all 14 mice (minimum 0.818) but Ach does not
(WT59 0.494, WT62 0.660). Shuffled labels give 0.49 to 0.54.

## fig1g-history-stay, fig1h-switch-no-history-post and fig1h-switch-da-approach

> In trials that repeat the previous port, lose-win trials have a larger mean DA and delta-Ach after side entry than win-win
> trials, and win-lose trials have a deeper DA dip and a larger delta-Ach than lose-lose trials.

> When the mouse switches ports, the previous outcome does not change DA or Ach after side-port entry.

> In switch trials, DA during the move from centre to side port is larger when the previous trial was unrewarded.

In [ ]:
# fig1g-history-stay, fig1h-switch-no-history-post, fig1h-switch-da-approach
W = lambda *ws: (lambda g: g.word.isin(ws))
comparisons = [  # claim, signal, measure, window, a, b, expected sign of a - b
    ("fig1g", "DA", "mean", "post", "aA", "AA", +1), ("fig1g", "Ach", "range", "post", "aA", "AA", +1),
    ("fig1g", "DA", "mean", "post", "Aa", "aa", -1), ("fig1g", "Ach", "range", "post", "Aa", "aa", +1),
    ("fig1h-absent", "DA", "mean", "post", "aB", "AB", 0), ("fig1h-absent", "Ach", "range", "post", "aB", "AB", 0),
    ("fig1h-absent", "DA", "mean", "post", "Ab", "ab", 0), ("fig1h-absent", "Ach", "range", "post", "Ab", "ab", 0),
    ("fig1h-approach", "DA", "mean", "pre", "aB+ab", "AB+Ab", +1)]
res = []
for claim, k, meas, which, a, b, sgn in comparisons:
    for rd in ("W1", "W2"):
        d = M[k][(M[k].reading == rd) & (M[k].which == which)]
        out, t2 = per_mouse(d, W(*a.split("+")), W(*b.split("+")), meas)
        sig = out.p < 0.05
        res.append(dict(claim=claim, signal=k, measure=meas, window=which, contrast=f"{a} - {b}", reading=rd, mice=len(out),
                        n_a_median=int(out.n_a.median()), n_b_median=int(out.n_b.median()),
                        sig_pos=int((sig & (out["diff"] > 0)).sum()), sig_neg=int((sig & (out["diff"] < 0)).sum()),
                        median_diff=float(out["diff"].median()), T2_p=float(t2)))
hist = pd.DataFrame(res)
print("T1 per mouse, a minus b (sig_pos / sig_neg = mice with p < 0.05 and a positive / negative difference):")
print(hist.round(4).to_string(index=False))

**Result: reproduced (fig1g-history-stay).** Under both windows, aA exceeds AA in mean DA in 14 of 14 mice and in delta-Ach in
9 of 16; Aa is below aa in mean DA in 14 of 14 and above it in delta-Ach in 10 of 16 (W1) and 9 of 16 (W2), with one mouse
the other way under W2. T2 p ≤ 0.017 for all four.

**Result: reproduced (fig1h-switch-no-history-post).** In switch trials the most mice with a significant difference in any
comparison is 5 of 14 (DA, aB against AB, W1: 1 higher, 4 lower; median -0.17 z, T2 p = 0.015); Ach aB against AB has none,
and Ab against ab has at most 2 mice for either signal. The AB words are rare (median 16 trials per mouse; 9 for Ach).

**Result: reproduced (fig1h-switch-da-approach).** Mean DA before side entry is higher after a loss in 13 of 14 mice (W1) and
14 of 14 (W2) with T1 p < 0.05, none the other way (median +0.25 z).

## ed3-not-lateralized

> DA and Ach transients do not differ between trials to the port ipsilateral and contralateral to the fibre; DA shows no
> significant difference and Ach differences are inconsistent in direction with poor LDA accuracy.

In [ ]:
# ed3-not-lateralized: ipsilateral against contralateral trials (fibre hemisphere from the channel metadata), rewarded and
# unrewarded separately, post window. DA: the Fig. 1 DA channels; Ach: the Fig. 1 Ach channels.
res = []; per = {}
for k, meas in (("DA", "mean"), ("Ach", "range")):
    m = M[k].copy()
    m["ipsi"] = np.where(m.hemi == "left", m.left, ~m.left)
    for rd in ("W1", "W2"):
        for rew in (True, False):
            d = m[(m.reading == rd) & (m.which == "post") & (m.rew == rew)]
            out, t2 = per_mouse(d, lambda g: g.ipsi, lambda g: ~g.ipsi, meas)
            per[(k, rd, rew)] = out
            sig = out.p < 0.05
            res.append(dict(signal=k, reading=rd, rewarded=rew, mice=len(out), sig_ipsi_higher=int((sig & (out["diff"] > 0)).sum()),
                            sig_contra_higher=int((sig & (out["diff"] < 0)).sum()), median_diff=float(out["diff"].median()),
                            max_abs_diff=float(out["diff"].abs().max()), T2_p=float(t2)))
lat = pd.DataFrame(res)
print("Ipsilateral minus contralateral, T1 per mouse:"); print(lat.round(4).to_string(index=False))
print("\nhemisphere of the Fig. 1 channels (mice):", {k: M[k].groupby("mouse").hemi.first().value_counts().to_dict() for k in M})
# LDA ipsi against contra, Ach, post W1
rng = np.random.default_rng(109); rows = []
for k in ("Ach", "DA"):
    for mouse in sorted(M[k].mouse.unique()):
        for rew in (True, False):
            X, y = [], []
            for s in [s for s in f1 if s["mouse"] == mouse and fig1_channel(s, k)]:
                ch = fig1_channel(s, k); t = achda.photometry_trials(s); t = t[t.was_rewarded == rew]
                seg, keep = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, "post", "W1"))
                ipsi = np.where(s["meta"][ch]["hemisphere"] == "left", t.chose_left.values, ~t.chose_left.values)[keep]
                X.append(seg); y.append(ipsi)
            X, y = np.vstack(X), np.concatenate(y)
            rows.append(dict(signal=k, mouse=mouse, rewarded=rew, acc=achda.lda_accuracy(X[y], X[~y], rng)))
latlda = pd.DataFrame(rows)
print("\nLDA accuracy, ipsilateral against contralateral (post W1):"); print(latlda.groupby(["signal", "rewarded"]).acc.agg(["mean", "min", "max"]).round(3).to_string())

**Result: not reproduced.** The criterion requires that no DA mouse differ; in rewarded trials 10 of 14 DA mice differ
between ipsilateral and contralateral trials with T1 p < 0.05 under W1 (7 higher ipsilateral, 3 lower) and 10 under W2, and
6 and 7 in unrewarded trials. The differences are small (largest 0.19 to 0.20 z, median 0.06 z in rewarded trials) and T2
across mice is not significant (p 0.07 to 0.32). The Ach part holds: significant differences go both ways (for example 3 and
4 mice under W1, rewarded), and the ipsilateral-contralateral LDA accuracy is 0.648 (rewarded) and 0.619 (unrewarded), below
0.65. The paper's Extended Data Fig. 3 uses 7 DA mice, which cannot be identified; all 14 are used here.

## fig1f-cue-omission

> In rewarded trials with the LED cues omitted, DA after side entry is larger and DA before side entry smaller than with
> cues, while Ach changes in the opposite direction.

In [ ]:
# fig1f-cue-omission: session pairs from aa_processStats_Fig1_DA_Ach_LED.m (cue-omission session listed first), the four mice
# whose channel 1 holds dLight1.1 and channel 5 Ach3.0 (channelsToAnalyze = [1 5] in that script).
LED_PAIRS = {"WT62": ("20211122", "20211108"), "WT63": ("20211122", "20211110"), "WT64": ("20211122", "20211108"), "WT65": ("20211122", "20211108")}
res = []
for mouse, (nocue, cue) in LED_PAIRS.items():
    sn, sc = key[(mouse, nocue)], key[(mouse, cue)]
    for k, ch in (("DA", "greenR"), ("Ach", "greenL")):
        for rd in ("W1", "W2"):
            for which in ("post", "pre"):
                vals = {}
                for lab, s in (("nocue", sn), ("cue", sc)):
                    t = achda.photometry_trials(s); t = t[t.was_rewarded]
                    seg, keep = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, which, rd))
                    vals[lab] = seg
                for meas in (("mean",) if k == "DA" else ("min", "range")):
                    e, p = achda.t1(achda.measure(vals["nocue"], meas), achda.measure(vals["cue"], meas))
                    res.append(dict(mouse=mouse, signal=k, measure=meas, reading=rd, window=which, n_nocue=len(vals["nocue"]), n_cue=len(vals["cue"]), diff_nocue_minus_cue=e, p=p))
cue = pd.DataFrame(res)
print("Rewarded trials, cue omission minus cued session, T1 per mouse:")
print(cue.pivot_table(index=["signal", "measure", "reading", "window"], columns="mouse", values=["diff_nocue_minus_cue", "p"]).round(4).to_string())
want = {("DA", "mean", "post"): +1, ("DA", "mean", "pre"): -1, ("Ach", "min", "post"): -1, ("Ach", "min", "pre"): +1}
for rd in ("W1", "W2"):
    c = cue[cue.reading == rd]
    ok = pd.DataFrame({f"{k[0]} {k[1]} {k[2]}": c[(c.signal == k[0]) & (c.measure == k[1]) & (c.window == k[2])].set_index("mouse").apply(lambda r: (np.sign(r.diff_nocue_minus_cue) == v) and r.p < 0.05, axis=1) for k, v in want.items()})
    ok["all four"] = ok.all(axis=1)
    print(f"\n{rd}: stated direction with p < 0.05, per mouse"); print(ok.to_string())
centre_led = {m: (len(key[(m, n)]["events"]["center_LED"]) / len(key[(m, n)]["trials"]), len(key[(m, c)]["events"]["center_LED"]) / len(key[(m, c)]["trials"]),
                  len(key[(m, n)]["events"]["side_LED"]) / len(key[(m, n)]["trials"]), len(key[(m, c)]["events"]["side_LED"]) / len(key[(m, c)]["trials"])) for m, (n, c) in LED_PAIRS.items()}
print("\nLED onsets per trial (centre: no-cue, cued; side: no-cue, cued):", {m: tuple(round(x, 2) for x in v) for m, v in centre_led.items()})

**Result: reproduced.** In all 4 mice and under both windows, without cues mean DA after side entry is higher (+0.38 to
+0.57 z under W1), mean DA before side entry lower (-0.12 to -0.26 z), the Ach minimum after side entry lower (-0.22 to -0.61
z) and before it higher (+0.20 to +0.44 z), each with T1 p < 0.05. The cue-omission sessions are those of the paper's LED
script; nothing in the files marks them. Their LED channel still has onsets, at about twice the centre-LED rate per trial of
the cued sessions (2.2 to 3.9 against 1.1 to 1.8).

## ed4bc-iti-port-entries

> DA and Ach are modulated more around port entries in unrewarded trials than around the same port entries during the ITI;
> after side entry DA dips more and Ach rises more in unrewarded trials, with the opposite before side entry.

In [ ]:
# ed4bc-iti-port-entries: ITI side-port entries (between a trial's side exit and the next centre entry) at least 15 samples
# from any other port entry or exit (Supplementary Methods, ITI analysis), against side entries of unrewarded trials.
def iti_entries(s):
    t = s["trials"]; ok = t.photometry_side_out_index.values >= 0
    ent = np.sort(np.r_[s["events"]["left_port"], s["events"]["right_port"]])
    allev = np.sort(np.r_[ent, s["events"]["center_port"], s["exits"]["center_port"], s["exits"]["left_port"], s["exits"]["right_port"]])
    so = t.photometry_side_out_index.values[ok]; ci_next = np.r_[t.photometry_center_in_index.values[ok][1:], np.inf]
    keep = []
    for e in ent:
        j = np.searchsorted(so, e, side="right") - 1
        if j < 0 or not (so[j] < e < ci_next[j]):
            continue
        others = allev[allev != e]
        if np.min(np.abs(others - e)) >= achda.ITI_SEP:
            keep.append(e)
    return np.array(keep, dtype=int)

res = []; counts = []
for k, meas_list in (("DA", ("mean",)), ("Ach", ("min", "range"))):
    for s in f1:
        ch = fig1_channel(s, k)
        if ch is None:
            continue
        z = s["z"][ch]; t = achda.photometry_trials(s); t = t[~t.was_rewarded]
        ie = iti_entries(s)
        counts.append(dict(signal=k, mouse=s["mouse"], iti=len(ie), norew=len(t)))
        for rd in ("W1", "W2"):
            for which in ("pre", "post"):
                a, _ = achda.segments(z, achda.window_idx(t.photometry_side_in_index.values, which, rd, n=achda.WIN_ITI))
                b, _ = achda.segments(z, achda.window_idx(ie, which, rd, n=achda.WIN_ITI))
                for meas in meas_list:
                    res.append(pd.DataFrame({"signal": k, "mouse": s["mouse"], "reading": rd, "which": which, "measure": meas,
                                             "cond": ["norew"] * len(a) + ["iti"] * len(b), "v": np.r_[achda.measure(a, meas), achda.measure(b, meas)] if len(b) else achda.measure(a, meas)}))
iti = pd.concat(res, ignore_index=True)
cnt = pd.DataFrame(counts).groupby(["signal", "mouse"]).sum()
print("Isolated ITI side entries per mouse:"); print(cnt.groupby("signal").iti.describe().round(1).to_string())
rows = []
for (k, rd, which, meas), g in iti.groupby(["signal", "reading", "which", "measure"]):
    out, t2 = per_mouse(g, lambda x: x.cond == "norew", lambda x: x.cond == "iti", "v")
    sig = out.p < 0.05
    rows.append(dict(signal=k, measure=meas, reading=rd, window=which, mice=int(out.n_b.gt(1).sum()), sig_norew_higher=int((sig & (out["diff"] > 0)).sum()),
                     sig_norew_lower=int((sig & (out["diff"] < 0)).sum()), median_diff=float(out["diff"].median()), T2_p=float(t2)))
print("\nUnrewarded-trial side entries minus ITI side entries, T1 per mouse:"); print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: not reproduced.** Isolated ITI side entries are few (median 6 per DA mouse and 7.5 per Ach mouse, 1 to 53).
After side entry DA is lower in unrewarded trials in 7 (W1) and 8 (W2) of 14 DA mice with T1 p < 0.05 (median -0.67 z),
reaching more than half only under W2; before side entry DA is higher in 5 of 14. The Ach minimum does not differ
consistently: after side entry 1 mouse higher and 1 lower (W1); before it 0 higher and 5 lower, the opposite of the stated
direction. The low counts of isolated ITI entries limit every per-mouse test.

## Behaviour

Behaviour uses every trial of each session, including those before the photometry started. Per-mouse metrics follow
block_transition_stats.ipynb; the RFLR is fitted per mouse by maximum likelihood (helper docstring).

In [ ]:
# Behaviour tables for every session (all trials, including those before the photometry started)
B = pd.concat([achda.behaviour_table(s) for s in S], ignore_index=True)
B["p_high"] = B[["left_reward_prob", "right_reward_prob"]].max(axis=1)

def history2(tab):
    """Two-trial history before each trial (trial t-2 then t-1): A = the port of t-2, B = the other; upper case = rewarded."""
    out = []
    for _, t in tab.groupby("session"):
        c, r = t.choice.values, t.was_rewarded.values
        h = [None, None]
        for i in range(2, len(t)):
            a = "A" if r[i - 2] else "a"
            b = ("A" if r[i - 1] else "a") if c[i - 1] == c[i - 2] else ("B" if r[i - 1] else "b")
            h.append(a + b)
        out.append(pd.Series(h, index=t.index))
    return pd.concat(out)
B["hist2"] = history2(B)

def mouse_metrics(tab):
    """Per-mouse behaviour metrics (block_transition_stats.ipynb)."""
    rows = []
    for mouse, t in tab.groupby("mouse"):
        cur = achda.block_position_curves(t).groupby("block_pos")[["phigh", "pswitch"]].mean().reset_index()
        post = cur[cur.block_pos > 0]
        params, _ = achda.fit_rflr(achda.session_arrays(t))
        rows.append(dict(mouse=mouse, sessions=t.session.nunique(), trials=len(t), meanITI=t.iti.mean(), decisionTime=t.decision_time.mean(),
                         lose_switch=t[t.prev_rew == False].switch.mean(), win_switch=t[t.prev_rew == True].switch.mean(), meanLeft=t.choice.mean(),
                         reward_rate=t.groupby("session").was_rewarded.mean().mean(), tau_highport=achda.tau_highport(cur),
                         max_pswitch=post.pswitch.max(), alpha=params[0], beta=params[1], tau=params[2]))
    return pd.DataFrame(rows)

def compare(ma, mb, label_a, label_b):
    rows = []
    for col, test in (("meanITI", "t"), ("decisionTime", "t"), ("lose_switch", "t"), ("win_switch", "t"), ("meanLeft", "t"), ("reward_rate", "t"),
                      ("tau_highport", "w"), ("max_pswitch", "w"), ("alpha", "w"), ("beta", "w"), ("tau", "w")):
        a, b = ma[col].dropna(), mb[col].dropna()
        p = stats.ttest_ind(a, b).pvalue if test == "t" else stats.ranksums(a, b).pvalue
        rows.append(dict(metric=col, test="two-sided t-test" if test == "t" else "Wilcoxon rank-sum", **{f"mean_{label_a}": a.mean(), f"mean_{label_b}": b.mean()}, n_a=len(a), n_b=len(b), p=p))
    return pd.DataFrame(rows)
print("Reward probability of the high port, sessions:", B.groupby("session").p_high.first().value_counts().to_dict())

## ed1b-switch-after-block-transition

> After a block transition, mice transiently increase their probability of switching ports, which brings them to the new
> high-reward-probability port.

In [ ]:
# ed1b-switch-after-block-transition and ed1hj-rflr-predicts-switching on the Fig. 1 mice (the n = 8 of Extended Data Fig. 1 are not identified)
B1 = B[B.mouse.isin(figmice("Fig1")) & B.session.isin([f'{s["mouse"]}_{s["date"]}' for s in f1])]
cur = achda.block_position_curves(B1)
pm = cur.groupby(["session", "block_pos"])[["phigh", "pswitch"]].mean().reset_index()
pm["mouse"] = pm.session.str.split("_").str[0]
mm = pm.groupby(["mouse", "block_pos"])[["phigh", "pswitch"]].mean().reset_index()
mean_curve = mm.groupby("block_pos")[["phigh", "pswitch"]].agg(["mean", "sem"])
print(f"{B1.mouse.nunique()} mice, {B1.session.nunique()} sessions, {cur.groupby('session').block_pos.apply(lambda x: (x == 0).sum()).sum()} block transitions")
per = mm.groupby("mouse").apply(lambda g: pd.Series(dict(pre=g[(g.block_pos >= -10) & (g.block_pos < 0)].pswitch.mean(), post=g[(g.block_pos >= 0) & (g.block_pos < 10)].pswitch.mean(),
                                                           late=g[(g.block_pos >= 20) & (g.block_pos <= 30)].pswitch.mean())), include_groups=False)
per["tau_highport"] = [achda.tau_highport(mm[mm.mouse == m]) for m in per.index]
print("mice with P(switch) higher in the 10 trials after than before: %d of %d; lower at positions 20-30 than after: %d; tau_highport positive: %d" % (
    int((per.post > per.pre).sum()), len(per), int((per.late < per.post).sum()), int((per.tau_highport > 0).sum())))
print("\nPer mouse: mean P(switch) over 10 trials before, the first 10 after, and positions 20-30; tau_highport (trials):"); print(per.round(3).to_string())
mc = mm.groupby("block_pos")[["phigh", "pswitch"]].mean().reset_index()
print("\nMean over mice: P(switch) before %.3f, first 10 after %.3f, positions 20-30 %.3f; tau_highport of the mean curve %.2f trials; P(high) at position 0 %.3f, at 10 %.3f, at 30 %.3f" % (
    mc[mc.block_pos < 0].pswitch.mean(), mc[(mc.block_pos >= 0) & (mc.block_pos < 10)].pswitch.mean(), mc[(mc.block_pos >= 20)].pswitch.mean(), achda.tau_highport(mc),
    mc[mc.block_pos == 0].phigh.item(), mc[mc.block_pos == 10].phigh.item(), mc[mc.block_pos == 30].phigh.item()))
print("\nby high-port probability (sessions, mean P(switch) first 10 after / before):",
      {p: (g.session.nunique(), round(g[(g.block_pos >= 0) & (g.block_pos < 10)].pswitch.mean(), 3), round(g[g.block_pos < 0].pswitch.mean(), 3))
       for p, g in cur.merge(B1.groupby("session").p_high.first().rename("p").reset_index(), on="session").groupby("p")})
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for a, col in zip(ax, ("pswitch", "phigh")):
    a.errorbar(mean_curve.index, mean_curve[(col, "mean")], mean_curve[(col, "sem")]); a.axvline(0, ls=":"); a.set_xlabel("block position"); a.set_ylabel(col)
plt.tight_layout(); plt.show()
print("plotted ranges: P(switch) %.3f to %.3f, P(high) %.3f to %.3f" % (mean_curve[("pswitch", "mean")].min(), mean_curve[("pswitch", "mean")].max(), mean_curve[("phigh", "mean")].min(), mean_curve[("phigh", "mean")].max()))

**Result: reproduced.** On the 26 Fig. 1 mice (121 sessions, 696 transitions), the mean P(switch) over mice is 0.175 in the 10
trials before a transition, 0.216 in the first 10 after it and 0.174 at positions 20 to 30. P(high port) falls to 0.119 at the
transition and recovers to 0.793 by position 10; its time constant is 2.82 trials for the mean curve and positive in every
mouse. P(switch) rises after the transition in 21 of 26 mice. The paper's 8 mice of Extended Data Fig. 1 are not identified,
and most of these sessions have a high-port probability of 0.85 or 0.9, not 0.95.

## ed1hj-rflr-predicts-switching

> The three-parameter RFLR model (alpha, beta, tau) predicts the switching dynamics at block transitions and the conditional
> switch probabilities given the past two trials.

In [ ]:
# ed1hj-rflr-predicts-switching: model probabilities on the mice's own histories, (a) fit on 70% of each mouse's sessions and
# evaluated on the held-out 30%, (b) fit and evaluated on all sessions of the mouse.
rng = np.random.default_rng(114)
def predict(tab, params):
    out = []
    for _, t in tab.groupby("session"):
        p_left = achda.rflr_probs(params, t.choice.values.astype(int), t.was_rewarded.values.astype(int))
        t = t.copy(); t["p_left"] = p_left
        t["p_switch_model"] = np.where(t.prev_choice == 1, 1 - p_left, p_left); t.loc[t.prev_choice.isna(), "p_switch_model"] = np.nan
        t["p_high_model"] = np.where(t.high_left, p_left, 1 - p_left)
        out.append(t)
    return pd.concat(out)
preds = {}
for reading in ("a", "b"):
    parts = []
    for mouse, t in B1.groupby("mouse"):
        sess = t.session.unique()
        if reading == "a":
            if len(sess) < 2:
                continue
            test = rng.choice(sess, max(1, int(round(0.3 * len(sess)))), replace=False)
            train = [x for x in sess if x not in test]
        else:
            test = train = sess
        params, _ = achda.fit_rflr(achda.session_arrays(t[t.session.isin(train)]))
        parts.append(predict(t[t.session.isin(test)], params).assign(alpha=params[0], beta=params[1], tau=params[2]))
    preds[reading] = pd.concat(parts)
for reading, P in preds.items():
    h = P.dropna(subset=["hist2", "switch"]).groupby("hist2").agg(n=("switch", "size"), mouse=("switch", "mean"), model=("p_switch_model", "mean"))
    h["se"] = np.sqrt(h.mouse * (1 - h.mouse) / h.n); h["within_se"] = (h.model - h.mouse).abs() <= h.se
    print(f"\nReading ({reading}): {P.mouse.nunique()} mice, {P.session.nunique()} evaluated sessions. Conditional P(switch) by two-trial history:")
    print(h.round(4).to_string()); print("histories within the binomial s.e.:", int(h.within_se.sum()), "of", len(h))
    P = P.copy(); P["switch_f"] = P.switch.astype(float); P["high_f"] = P.high.astype(float)
    cm = achda.block_position_curves(P)
    # model curves: same positions, model probabilities
    rows = []
    for sess, t in P.groupby("session"):
        t = t.reset_index(drop=True)
        for st in t.index[(t.block_pos == 0) & ~t.first_block]:
            for k in range(-10, 31):
                i = st + k
                if 0 <= i < len(t) and ((k >= 0 and t.block[i] == t.block[st]) or (k < 0 and t.block[i] == t.block[st] - 1)):
                    rows.append((k, t.high_f[i], t.switch_f[i], t.p_high_model[i], t.p_switch_model[i]))
    c = pd.DataFrame(rows, columns=["block_pos", "phigh", "pswitch", "phigh_model", "pswitch_model"]).groupby("block_pos").mean()
    print("correlation across block positions -10..30, P(switch) mouse vs model r = %.3f; P(high) r = %.3f" % (c.pswitch.corr(c.pswitch_model), c.phigh.corr(c.phigh_model)))
    print("fitted parameters (median over mice): alpha %.2f, beta %.2f, tau %.2f" % tuple(P.groupby("mouse")[["alpha", "beta", "tau"]].first().median()))

**Result: not reproduced.** Under both readings the model's conditional P(switch) falls within the binomial s.e. of the mice's
for 1 of 8 histories (Aa); it overestimates AB (0.34 against 0.19 to 0.22) and underestimates aB and ab. Around transitions the
model follows P(high port) (r = 0.970 held out, 0.986 pooled) and P(switch) under reading (b) (r = 0.918) but not under reading
(a) (r = 0.803). With thousands of trials per history the s.e. is 0.002 to 0.03, so the criterion is strict.

## Behaviour by cohort

Cohorts: striatum-wide TelC and mCherry, and VLS TelC, from the virus names in the files; Drd2 genotypes from the export table;
cortical and thalamic TelC and mCherry from the export table's Vglut1-Cre and Vglut2-Cre labels and the virus names.

In [ ]:
# Cohorts from the CSV labels and the files' virus names
GROUPS = {
    "sw_TelC": ["S1194", "S1214", "S1247", "S1258", "S1259"],   # Fig3, SIO-tetanus in the left hemisphere, striatum-wide
    "sw_mCh": ["S1233", "S1234", "S1246", "S1248", "S1260"],    # Fig3, DIO mCherry both hemispheres
    "vls_TelC_dlight": ["S732", "S735", "S736", "S776"],        # Fig3, DIO tetanus in one hemisphere, mCherry in the other
    "vls_TelC_rdah": ["S950", "S952", "S1142", "S1143"],        # EDFig16, rDAh + Ach3.0
    "cko": figmice("Fig4", "Chat Jax D2R KO"), "ff": figmice("Fig4", "Chat Jax neg D2R f/f"),
    "wt": figmice("Fig4", "Chat jax") + figmice("Fig4", "Chat Jax"),
    "ctx_TelC": ["S1426", "S1428", "S1488"], "ctx_mCh": ["S1463", "S1464", "S1465", "S1466"],     # Vglut1Cre, retro tetanus / retro mCherry
    "thal_TelC": ["S1453", "S1454", "S1455", "S1486"], "thal_mCh": ["S1351", "S1389", "S1393"],  # Vglut2Cre, SIO-tetanus / mCherry
}
for g, mice in GROUPS.items():
    labs = {s["meta"][c]["sensor"] for s in S if s["mouse"] in mice for c in s["meta"] if s["meta"][c]["sensor"] != "unknown"}
    print(f"{g}: {len(mice)} mice, {sum(s['mouse'] in mice for s in S)} sessions; channel labels {sorted(labs)}")
MET = {g: mouse_metrics(B[B.mouse.isin(m)]) for g, m in GROUPS.items() if not g.startswith("vls_TelC_rdah")}

## fig3e-striatum-wide-telc

> Striatum-wide TelC in CINs causes severe behavioural defects (changed mean ITI, decision time and RFLR alpha), yet DA still
> has opposite polarity after side entry for rewarded and unrewarded trials.

In [ ]:
# fig3e-striatum-wide-telc, behaviour
cmp = compare(MET["sw_TelC"], MET["sw_mCh"], "TelC", "mCherry")
print(cmp[cmp.metric.isin(["meanITI", "decisionTime", "alpha"])].round(4).to_string(index=False))
print("\nper mouse:"); print(pd.concat([MET["sw_TelC"].assign(g="TelC"), MET["sw_mCh"].assign(g="mCherry")])[["g", "mouse", "sessions", "trials", "meanITI", "decisionTime", "alpha", "beta", "tau"]].round(3).to_string(index=False))
# DA in TelC mice: mean z-scored dLight in the post window, rewarded and unrewarded
rows = []
for s in S:
    if s["mouse"] not in GROUPS["sw_TelC"] + GROUPS["sw_mCh"]:
        continue
    t = achda.photometry_trials(s)
    for ch in achda.channel_of(s, {"dlight1.1"}):
        for rd in ("W1", "W2"):
            seg, keep = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, "post", rd))
            rows.append(pd.DataFrame({"mouse": s["mouse"], "ch": ch, "reading": rd, "rew": t.was_rewarded.values[keep], "v": seg.mean(1)}))
da3e = pd.concat(rows)
da3e["group"] = np.where(da3e.mouse.isin(GROUPS["sw_TelC"]), "TelC", "mCherry")
pmouse = da3e.groupby(["group", "reading", "rew", "mouse"]).v.mean().reset_index()
print("\nMean z-scored DA in the post window, mean over mice (per-mouse values below):")
print(pmouse.groupby(["group", "reading", "rew"]).v.agg(["mean", "min", "max", "count"]).round(3).to_string())
print("dLight channels used per TelC mouse:", da3e[da3e.group == "TelC"].groupby("mouse").ch.unique().to_dict())

**Result: reproduced.** TelC mice have longer ITIs (6.44 against 2.53 s, p = 0.0007; paper p = 0.023), longer decision times
(1.53 against 0.90 s, p < 0.0001) and a larger alpha (0.77 against 0.37, rank-sum p = 0.028; paper 0.047). In TelC mice mean
z-scored DA after side entry is +0.54 (W1) and +0.53 (W2) in rewarded and -0.36 and -0.32 in unrewarded trials, in every
TelC mouse.

## ed9jm-vls-telc-behaviour-normal, ed11fh-cko-general-behaviour-comparable, ed11i-cko-rflr and ed13-input-telc-behaviour

In [ ]:
# ed9jm-vls-telc-behaviour-normal, reading (a): VLS-TelC cohort against the striatum-wide mCherry mice
print("ed9jm, reading (a):"); print(compare(MET["vls_TelC_dlight"], MET["sw_mCh"], "VLS_TelC", "mCherry").round(4).to_string(index=False))
# ed11fh / ed11i: Drd2-cKO against each control
for ctrl in ("ff", "wt"):
    print(f"\nDrd2-cKO against {ctrl}:"); print(compare(MET["cko"], MET[ctrl], "cKO", ctrl).round(4).to_string(index=False))
both = pd.concat([MET["ff"], MET["wt"]])
print("\nDrd2-cKO against both controls pooled (RFLR):"); c = compare(MET["cko"], both, "cKO", "ctrl"); print(c[c.metric.isin(["beta", "tau", "alpha"])].round(4).to_string(index=False))
# ed13: cortical and thalamic TelC against mCherry
for inp in ("ctx", "thal"):
    print(f"\n{inp} TelC against mCherry:"); print(compare(MET[f"{inp}_TelC"], MET[f"{inp}_mCh"], "TelC", "mCherry").round(4).to_string(index=False))

> ed9jm: Mice with VLS-selective TelC in CINs show no behavioural deficits.

**Result: not reproduced (ed9jm-vls-telc-behaviour-normal).** Only reading (a) can be run: against the striatum-wide mCherry
mice, the VLS-TelC mice differ in decision time (p = 0.0029), lose-switch (p = 0.035), win-switch (p = 0.0096) and reward
rate (p = 0.0063); alpha and beta give rank-sum p = 0.050. Reading (b) needs sessions before expression, which the dataset
does not label; reading (c) cannot be formed, as every session records both hemispheres. The bootstrapped conditional switch
probabilities were not run, as the t-tests already find differences.

> ed11fh: General performance metrics and block-transition dynamics of Drd2-cKO mice do not differ from Drd2 f/f and WT mice.

**Result: not reproduced (ed11fh-cko-general-behaviour-comparable).** Of the 14 tests (7 metrics against each control), one is
significant: decision time, Drd2-cKO 1.52 against Drd2 f/f 1.30 s, p = 0.044. Against WT none is (smallest p = 0.11).

> ed11i: The Extended Data Fig. 11i legend reports significant differences in the RFLR beta and tau of Drd2-cKO mice,
> although the main text calls the RFLR coefficients comparable.

**Result: not reproduced (ed11i-cko-rflr).** Beta and tau do not differ under any reading: against Drd2 f/f p = 0.064 and
0.42, against WT 0.64 and 0.64, against both pooled 0.24 and 0.49. The main text's "comparable" is what the data show.

> ed13: Cortical or thalamic TelC impairs performance, with lower reward rate and win-switch differences, and the RFLR shows
> a smaller beta and a larger tau.

**Result: partially reproduced (ed13-input-telc-behaviour).** Thalamic TelC mice (4, against 3 mCherry) have a lower beta (1.18
against 3.29, p = 0.034), a larger tau (1.69 against 0.39, p = 0.034), a lower reward rate (0.648 against 0.819, p = 0.030)
and a higher win-switch (0.30 against 0.06, p = 0.0016). Cortical TelC mice (3 of the paper's 5, against 4 mCherry) differ in
none (beta p = 0.72, tau p = 0.48, reward rate p = 0.66). Each of these mice has one session in the dataset.

## ed11j-cko-switching-impaired

> Drd2-cKO mice are impaired in switching ports after some action-outcome histories compared with both Drd2 f/f and WT mice.

In [ ]:
# ed11j-cko-switching-impaired: bootstrapped difference of conditional P(switch) (1000 resamples, Supplementary Methods)
rng = np.random.default_rng(118)
H2 = ["AA", "Aa", "aA", "aa", "AB", "Ab", "aB", "ab"]
def history3(tab):
    out = []
    for _, t in tab.groupby("session"):
        c, r = t.choice.values, t.was_rewarded.values; h = [None] * min(3, len(t))
        for i in range(3, len(t)):
            ref = c[i - 3]; s = ""
            for j in (i - 3, i - 2, i - 1):
                l = "A" if c[j] == ref else "B"; s += l if r[j] else l.lower()
            h.append(s)
        out.append(pd.Series(h, index=t.index))
    return pd.concat(out)
B["hist3"] = history3(B)
def counts(tab, col):
    d = tab.dropna(subset=[col, "switch"])
    g = d.groupby(["session", col]).switch.agg(["sum", "size"]).unstack(col, fill_value=0)
    return g
res = []
for nback, col in ((2, "hist2"), (3, "hist3")):
    C = {g: counts(B[B.mouse.isin(GROUPS[g])], col) for g in ("cko", "ff", "wt")}
    hists = sorted(set.intersection(*[set(C[g]["size"].columns) for g in C]))
    for unit in ("trials", "sessions"):
        boot = {}
        for g, c in C.items():
            sw, n = c["sum"][hists].values, c["size"][hists].values
            if unit == "sessions":
                idx = rng.integers(0, len(sw), (1000, len(sw)))
                boot[g] = sw[idx].sum(1) / np.maximum(n[idx].sum(1), 1)
            else:
                tot_sw, tot_n = sw.sum(0), n.sum(0)
                boot[g] = rng.binomial(tot_n[None, :], (tot_sw / np.maximum(tot_n, 1))[None, :], (1000, len(hists))) / np.maximum(tot_n, 1)
        for ctrl in ("ff", "wt"):
            d = boot["cko"] - boot[ctrl]
            lo, hi = np.percentile(d, [2.5, 97.5], axis=0)
            for j, h in enumerate(hists):
                res.append(dict(nback=nback, unit=unit, control=ctrl, history=h, n_cko=int(C["cko"]["size"][h].sum()),
                                p_cko=C["cko"]["sum"][h].sum() / C["cko"]["size"][h].sum(), p_ctrl=C[ctrl]["sum"][h].sum() / C[ctrl]["size"][h].sum(),
                                ci_lo=lo[j], ci_hi=hi[j], lower_sig=hi[j] < 0))
cs = pd.DataFrame(res)
w = cs.pivot_table(index=["nback", "unit", "history"], columns="control", values="lower_sig", aggfunc="first")
w["both"] = w.ff & w.wt
print("Histories with cKO P(switch) lower than the control, 95% CI of the difference below zero:")
print(w.groupby(level=[0, 1]).both.agg(["sum", "size"]).to_string())
print(w[w.both].to_string())
print("\nTwo-trial histories, trial resampling:"); print(cs[(cs.nback == 2) & (cs.unit == "trials")].round(3).to_string(index=False))

**Result: not reproduced, resting on an assumption.** Resampling sessions, as the paper's bootstrap code does
(`resampled_means` in block_transition_stats.ipynb), no two- or three-trial history has a 95% interval of the Drd2-cKO minus
control difference below zero against both controls. Resampling trials, one history does (Aa: 0.516 against 0.597 and 0.616;
AAa among three-trial histories). The outcome takes the code's unit.

## fig2e-trial-and-noise-correlation

> Trial-segregated DA and Ach signals are anticorrelated with an offset of about 100 ms in rewarded and unrewarded trials, and
> their noise components show a similar structure.

In [ ]:
# Fig. 2: simultaneous rDAh (redL) and Ach3.0 (greenL) in the left hemisphere of the six Fig2 mice
F2 = figmice("Fig2")
f2 = sessions_of(F2, "Fig2")
MAXLAG = 20
def trial_xc(sessions, ch_a, ch_b, rew=None, noise=False, align="center"):
    """Trial-segregated normalized cross-correlation, one mouse: segments of 40 before and 100 after the alignment event,
    concatenated over trials and sessions (noise: each trial minus the mean trial of the mouse)."""
    A, Bm = [], []
    for s in sessions:
        if ch_a not in s["z"] or ch_b not in s["z"]:
            continue
        t = achda.photometry_trials(s)
        if rew is not None:
            t = t[t.was_rewarded == rew]
        ev = (t.photometry_center_in_index if align == "center" else t.photometry_side_in_index).values
        a, keep = achda.trial_matrix(s["z"][ch_a], ev); b, _ = achda.trial_matrix(s["z"][ch_b], ev)
        A.append(a); Bm.append(b)
    if not A:
        return None
    A, Bm = np.vstack(A), np.vstack(Bm)
    if noise:
        A, Bm = A - A.mean(0), Bm - Bm.mean(0)
    return achda.xcorr_norm(A.ravel(), Bm.ravel(), MAXLAG)
def xc_summary(xcs, kind="min"):
    lags = xcs[0][0]; R = np.array([r for _, r in xcs]); mean = R.mean(0)
    i = mean.argmin() if kind == "min" else mean.argmax()
    per = [(lags[r.argmin()] if kind == "min" else lags[r.argmax()], r.min() if kind == "min" else r.max()) for r in R]
    return lags, mean, lags[i], mean[i], per
# fig2e-trial-and-noise-correlation
rows = []; XC = {}
for align in ("center", "side"):
    for rew in (True, False):
        for noise in (False, True):
            xcs = [trial_xc([s for s in f2 if s["mouse"] == m], "redL", "greenL", rew, noise, align) for m in F2]
            xcs = [x for x in xcs if x is not None]
            lags, mean, lag, val, per = xc_summary(xcs)
            XC[(align, rew, noise)] = (lags, mean)
            rows.append(dict(align=align, rewarded=rew, component="noise" if noise else "signal", mice=len(xcs), lag_of_min_samples=int(lag), lag_s=lag * achda.DT,
                             min_r=val, per_mouse_lags=[int(p[0]) for p in per], per_mouse_min=[round(p[1], 3) for p in per]))
fig2e = pd.DataFrame(rows)
print("Mean over mice of the normalized cross-correlation (positive lag = DA leads Ach):"); print(fig2e.round(3).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for (align, rew, noise), (lags, mean) in XC.items():
    if align == "center":
        ax[int(noise)].plot(lags * achda.DT, mean, label="rewarded" if rew else "unrewarded")
for a, ttl in zip(ax, ("signal", "noise")):
    a.axvline(0, ls=":"); a.set_title(ttl); a.set_xlabel("lag (s), DA leads Ach >0"); a.legend()
plt.tight_layout(); plt.show()
print("plotted range of r: %.3f to %.3f" % (min(m.min() for _, m in XC.values()), max(m.max() for _, m in XC.values())))

**Result: reproduced.** Centre-aligned, the mean cross-correlation over the 6 mice has its minimum at +1 sample (0.054 s) in
rewarded trials (signal r = -0.226, noise -0.237) and at +2 samples (0.108 s) in unrewarded trials (-0.239 and -0.223), all
within 0.1 ± 0.054 s. Per mouse the minimum is at 0 to 2 samples. Side-aligned segments, the code's variant, give the same lags.

## fig2d-anticorrelated-lag

> Simultaneously recorded DA and Ach in the same hemisphere are highly anticorrelated over whole sessions, with the minimum at
> a positive lag (Ach changes follow DA changes after a short delay).

In [ ]:
# fig2d-anticorrelated-lag: whole-session cross-correlation (54-ms samples; the 18-ms demodulation is not in the files)
rng = np.random.default_rng(120)
rows = []
for m in F2:
    ss = [s for s in f2 if s["mouse"] == m and "redL" in s["z"] and "greenL" in s["z"]]
    for i, s in enumerate(ss):
        da, ach = s["z"]["redL"], s["z"]["greenL"]
        lags, r = achda.xcorr_norm(da, ach, MAXLAG)
        o = ss[(i + 1) % len(ss)]
        n = min(len(o["z"]["redL"]), len(ach))
        _, r_other = achda.xcorr_norm(o["z"]["redL"][:n], ach[:n], MAXLAG) if len(ss) > 1 else (None, np.full_like(r, np.nan))
        _, r_shift = achda.xcorr_norm(np.roll(da, rng.integers(1, 10001)), ach, MAXLAG)
        rows.append(dict(mouse=m, session=s["date"], r=r, r_other=r_other, r_shift=r_shift))
d2 = pd.DataFrame(rows)
mouse_mean = {k: np.array([np.mean(np.vstack(g[k].values), 0) for _, g in d2.groupby("mouse")]) for k in ("r", "r_other", "r_shift")}
mean = {k: np.nanmean(v, 0) for k, v in mouse_mean.items()}
i = mean["r"].argmin()
print(f"{d2.mouse.nunique()} mice, {len(d2)} sessions. Mean over mice: minimum r = {mean['r'][i]:.3f} at lag {lags[i]} samples ({lags[i] * achda.DT:.3f} s); "
      f"maximum r = {mean['r'].max():.3f} at lag {lags[mean['r'].argmax()]}")
print(f"controls: other session of the same mouse, minimum {np.nanmin(mean['r_other']):.3f} (value at that lag {mean['r_other'][i]:.3f}); "
      f"circularly shifted DA, minimum {mean['r_shift'].min():.3f} (at that lag {mean['r_shift'][i]:.3f})")
print("per mouse, lag of the minimum (samples) and its value:", {m: (int(lags[v.argmin()]), round(v.min(), 3)) for m, v in zip(sorted(d2.mouse.unique()), mouse_mean["r"])})

**Result: reproduced.** Over whole sessions the mean cross-correlation has its minimum, r = -0.224, at +2 samples (0.108 s),
below 0.3 s; both controls stay near zero (minima -0.001). The files hold only the 54-ms demodulation, not the 18-ms one of
Fig. 2d, so the lag is resolved to 54 ms.

## fig2a-rdah-smaller-than-dlight, ed6b-separate-hemispheres and ed6c-da-bilateral

> fig2a: DA signals recorded with rDAh and dLight1.1 in opposite hemispheres of the same mice are comparable in shape, with
> consistently smaller amplitudes for rDAh.

> ed6b: DA and Ach recorded in separate hemispheres are also anticorrelated at a positive lag, with a smaller positive
> correlation at a negative lag.

> ed6c: DA release is highly correlated across the two hemispheres of the same mouse.

In [ ]:
# fig2a-rdah-smaller-than-dlight and ed6b-separate-hemispheres: WT63-WT65, dLight1.1 in the right hemisphere (greenR),
# rDAh (redL) and Ach3.0 (greenL) in the left; ed6c-da-bilateral: WT66-WT69, dLight1.1 in both hemispheres
OPP = [m for m in F2 if all(s["meta"].get("greenR", {}).get("sensor") == "dlight1.1" for s in f2 if s["mouse"] == m)]
print("mice with dLight1.1 and rDAh in opposite hemispheres:", OPP)
rows = []
for m in OPP:
    ss = [s for s in f2 if s["mouse"] == m and "greenR" in s["z"] and "redL" in s["z"]]
    for rd in ("W1", "W2"):
        for rew in (True, False):
            v = {}
            for ch in ("greenR", "redL"):
                segs = []
                for s in ss:
                    t = achda.photometry_trials(s); t = t[t.was_rewarded == rew]
                    seg, _ = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, "post", rd)); segs.append(seg)
                v[ch] = np.vstack(segs)
            meas = "max" if rew else "min"
            a, b = achda.measure(v["redL"], meas), achda.measure(v["greenR"], meas)
            e, p = achda.t1(np.abs(a), np.abs(b))
            rows.append(dict(mouse=m, reading=rd, rewarded=rew, measure=meas, rdah=a.mean(), dlight=b.mean(), abs_rdah_minus_dlight=e, p=p))
    # trial-averaged traces, -40..+100 about centre entry, rewarded and unrewarded
    for rew in (True, False):
        tr = {ch: np.vstack([achda.trial_matrix(s["z"][ch], achda.photometry_trials(s)[achda.photometry_trials(s).was_rewarded == rew].photometry_center_in_index.values)[0] for s in ss]).mean(0) for ch in ("greenR", "redL")}
        rows.append(dict(mouse=m, reading="trace", rewarded=rew, measure="r(mean traces)", rdah=np.nan, dlight=np.nan, abs_rdah_minus_dlight=np.corrcoef(tr["greenR"], tr["redL"])[0, 1], p=np.nan))
f2a = pd.DataFrame(rows)
print(f2a.round(4).to_string(index=False))
# ed6b: dLight (greenR) against Ach (greenL), opposite hemispheres, trial-segregated
xcs = [trial_xc([s for s in f2 if s["mouse"] == m], "greenR", "greenL") for m in OPP]
lags, mean, lag, val, per = xc_summary(xcs)
imax = mean[lags < 0].argmax()
print(f"\ned6b: mean over {len(xcs)} mice, minimum r = {val:.3f} at lag {lag} samples; maximum at negative lags r = {mean[lags < 0][imax]:.3f} at lag {lags[lags < 0][imax]}; per mouse minima {per}")
# ed6c
BIL = ["WT66", "WT67", "WT68", "WT69"]
rows = []
for m in BIL:
    ss = [s for s in S if s["mouse"] == m]
    print(m, "sessions", len(ss), "channels", sorted({(c, s["meta"][c]["sensor"], s["meta"][c]["brainRegion"]) for s in ss for c in s["z"]}))
    lg, r = trial_xc(ss, "greenR", "greenL")
    rows.append(dict(mouse=m, peak_r=r.max(), lag_of_peak=int(lg[r.argmax()])))
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Result: partially reproduced (fig2a-rdah-smaller-than-dlight).** In rewarded trials the rDAh maximum after side entry is
smaller than the dLight1.1 maximum in all 3 mice (by 0.64 to 1.03 z, p < 0.0001). In unrewarded trials the rDAh minimum is
deeper in WT63 (+0.28 z) and WT64 (+0.47 z) and does not differ in WT65. The trial-averaged traces correlate at 0.79 to 0.95
(WT65 rewarded 0.788).

**Result: reproduced (ed6b-separate-hemispheres).** dLight1.1 (right) against Ach3.0 (left) in WT63 to WT65: the mean has a
minimum of -0.405 at +2 samples and, at negative lags, a maximum of +0.138 at -6 samples; per mouse the minima are -0.36 to
-0.45 at +2 to +3 samples.

**Result: not reproduced (ed6c-da-bilateral).** The peak cross-correlation of the two dLight1.1 hemispheres is 0.633, 0.669,
0.679 and 0.488, at lag 0 in every mouse: positive and synchronous, but below the 0.7 the criterion names for "highly".

## fig2fg-off-diagonal

> The two-dimensional covariance of DA and Ach fluctuations has a negative off-diagonal at which DA changes precede Ach changes
> by about 100 ms at most times, and it nearly disappears when the mouse enters the side port.

In [ ]:
# fig2fg-off-diagonal: K(t1, t2) = mean over trials of residual DA(t1) x residual Ach(t2) (Supplementary Methods equation)
def offdiag(sessions, rew, lag, align, ch_da="redL", ch_ach="greenL"):
    A, Bm, side_rel = [], [], []
    for s in sessions:
        if ch_da not in s["z"] or ch_ach not in s["z"]:
            continue
        t = achda.photometry_trials(s); t = t[t.was_rewarded == rew]
        ev = (t.photometry_center_in_index if align == "center" else t.photometry_side_in_index).values
        a, keep = achda.trial_matrix(s["z"][ch_da], ev); b, _ = achda.trial_matrix(s["z"][ch_ach], ev)
        A.append(a); Bm.append(b); side_rel.append((t.photometry_side_in_index - t.photometry_center_in_index).values[keep])
    if not A:
        return None
    A, Bm = np.vstack(A), np.vstack(Bm)
    A, Bm = A - A.mean(0), Bm - Bm.mean(0)
    K = A.T @ Bm / len(A)          # K[t1, t2]
    n = K.shape[0]
    od = np.array([K[i, i + lag] for i in range(n - lag)])
    return od, int(np.median(np.concatenate(side_rel))), K
LAG = {rew: int(fig2e.loc[(fig2e["align"] == "center") & (fig2e.rewarded == rew) & (fig2e.component == "signal"), "lag_of_min_samples"].iloc[0]) for rew in (True, False)}
print("lag of the off-diagonal (samples, from fig2e):", LAG)
rows = []; OD = {}
for rew in (True, False):
    for align in ("side", "center"):
        res = [offdiag([s for s in f2 if s["mouse"] == m], rew, LAG[rew], align) for m in F2]
        res = [r for r in res if r is not None]
        od = np.mean([r[0] for r in res], 0); OD[(rew, align)] = od
        se_pos = achda.XC_PRE if align == "side" else achda.XC_PRE + int(np.median([r[1] for r in res]))
        near = np.abs(od[se_pos - 1: se_pos + 2]).mean()
        rows.append(dict(rewarded=rew, reading="(a) side-aligned" if align == "side" else "(b) centre-aligned, median side entry", lag=LAG[rew], mice=len(res),
                         side_entry_sample=se_pos, frac_negative=(od < 0).mean(), median_abs=np.median(np.abs(od)), abs_near_side_entry=near,
                         ratio=near / np.median(np.abs(od))))
f2fg = pd.DataFrame(rows)
print(f2fg.round(4).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for i, align in enumerate(("side", "center")):
    for rew in (True, False):
        ax[i].plot((np.arange(len(OD[(rew, align)])) - achda.XC_PRE) * achda.DT, OD[(rew, align)], label="rewarded" if rew else "unrewarded")
    ax[i].axvline(0, ls=":"); ax[i].set_title(f"off-diagonal, {align}-aligned"); ax[i].set_xlabel("t1 (s)"); ax[i].legend()
plt.tight_layout(); plt.show()
print("plotted range: %.3f to %.3f" % (min(v.min() for v in OD.values()), max(v.max() for v in OD.values())))

**Result: partially reproduced.** The off-diagonal, at the lag of fig2e (1 sample rewarded, 2 unrewarded), is negative at every
time point under both readings. Its magnitude within one sample of side entry is 0.22 (side-aligned) and 0.30
(centre-aligned) of its median in unrewarded trials, and 0.54 and 1.34 in rewarded trials; it falls below a quarter only in
unrewarded, side-aligned trials.

## ed11e-cko-anticorrelation-reduced

> In Drd2-cKO mice the anticorrelation between DA and Ach is significantly smaller in magnitude, and the off-diagonal negative
> covariance is reduced.

In [ ]:
# ed11e-cko-anticorrelation-reduced: per mouse, minimum of the trial-segregated cross-correlation (rDAh redL, Ach3.0 greenL)
# and mean off-diagonal (side-aligned, lag of fig2e), Drd2-cKO against each control
rows = []
for g in ("cko", "ff", "wt"):
    for m in GROUPS[g]:
        ss = [s for s in S if s["mouse"] == m and (m, s["date"]) in set(zip(uses[uses.Figure == "Fig4"].Mouse, uses[uses.Figure == "Fig4"].date))]
        for rew in (True, False):
            for noise in (False, True):
                x = trial_xc(ss, "redL", "greenL", rew, noise)
                if x is None:
                    continue
                od = offdiag(ss, rew, LAG[rew], "side")
                rows.append(dict(group=g, mouse=m, rewarded=rew, component="noise" if noise else "signal", sessions=sum("redL" in s["z"] and "greenL" in s["z"] for s in ss),
                                 min_r=x[1].min(), lag=int(x[0][x[1].argmin()]), offdiag_mean=od[0].mean()))
e11 = pd.DataFrame(rows)
print("mice with both channels:", e11.groupby("group").mouse.nunique().to_dict())
out = []
for (rew, comp), d in e11.groupby(["rewarded", "component"]):
    for ctrl in ("ff", "wt"):
        a, b = d[d.group == "cko"], d[d.group == ctrl]
        out.append(dict(rewarded=rew, component=comp, control=ctrl, min_r_cko=a.min_r.mean(), min_r_ctrl=b.min_r.mean(), p_min_r=stats.ttest_ind(a.min_r, b.min_r).pvalue,
                        offdiag_cko=a.offdiag_mean.mean(), offdiag_ctrl=b.offdiag_mean.mean(), p_offdiag=stats.ttest_ind(a.offdiag_mean, b.offdiag_mean).pvalue))
print(pd.DataFrame(out).round(4).to_string(index=False))

**Result: reproduced.** The minimum of the cross-correlation is -0.05 to -0.15 in Drd2-cKO mice against -0.26 to -0.33 in the
controls, for signal and noise in both trial types, with p ≤ 0.0092 against each control; the mean off-diagonal is -0.08
(unrewarded) and -0.12 (rewarded) against -0.22 to -0.25, p ≤ 0.011.

## fig3f-vls-telc-da-unchanged

> Blocking Ach release from VLS CINs in one hemisphere does not change DA dynamics there compared with the control hemisphere,
> in any trial type or history up to three trials back.

In [ ]:
# VLS-selective TelC: the TelC hemisphere of each mouse from the virus names (DIO tetanus / SIO-tetanus / 301 = TelC,
# DIO mCherry / 304 = mCherry). In S1142 and S1143 the left channels carry virus 222 and the right 304 (mCherry), so the left
# hemisphere is taken as TelC (an assumption: the files do not name virus 222).
TELC_SIDE = {}
for s in S:
    if s["mouse"] in GROUPS["vls_TelC_dlight"] + GROUPS["vls_TelC_rdah"]:
        labs = {c: s["meta"][c]["sensor"] + " " + s["meta"][c]["virus"] for c in s["meta"]}
        left = " ".join(v for c, v in labs.items() if c.endswith("L")); right = " ".join(v for c, v in labs.items() if c.endswith("R"))
        side = "L" if ("tetanus" in left or "301" in left or "222" in left) else "R" if ("tetanus" in right or "301" in right) else None
        TELC_SIDE.setdefault(s["mouse"], set()).add(side)
print("TelC hemisphere per mouse:", TELC_SIDE)
TELC_SIDE = {m: v.pop() for m, v in TELC_SIDE.items()}

def word_n(t, n):
    """History string of trials t-n..t, A = the port of trial t-n, upper case = rewarded (previous trials first)."""
    c, r = t.choice.values, t.was_rewarded.values; out = [None] * len(t)
    for i in range(n, len(t)):
        ref = c[i - n]; s = ""
        for j in range(i - n, i + 1):
            l = "A" if c[j] == ref else "B"; s += l if r[j] else l.lower()
        out[i] = s
    return out

def hemi_rows(mice, da_chan, n_back=(1, 2, 3)):
    """Per trial, mean z-scored DA pre and post (W1, W2) in the TelC and mCherry hemispheres, with history strings."""
    rows = []
    for s in S:
        if s["mouse"] not in mice:
            continue
        ts = TELC_SIDE[s["mouse"]]; chans = {"TelC": da_chan + ts, "mCherry": da_chan + ("R" if ts == "L" else "L")}
        if not all(c in s["z"] for c in chans.values()):
            continue
        t0 = add = achda.add_history(s["trials"])
        for n in n_back:
            t0[f"h{n}"] = word_n(t0, n)
        t = t0[t0.has_all_photometry_data & (t0.photometry_side_in_index >= 0)]
        for hemi, ch in chans.items():
            for rd in ("W1", "W2"):
                for which in ("pre", "post"):
                    seg, keep = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, which, rd))
                    tk = t[keep]
                    rows.append(pd.DataFrame({"mouse": s["mouse"], "hemi": hemi, "reading": rd, "which": which, "rew": tk.was_rewarded.values,
                                              **{f"h{n}": tk[f"h{n}"].values for n in n_back}, "v": seg.mean(1)}))
    return pd.concat(rows, ignore_index=True)

def hemi_tests(H):
    res = []
    conds = [("all", None, None), ("rewarded", "rew", True), ("unrewarded", "rew", False)]
    for n in (1, 2, 3):
        for h in sorted(H[f"h{n}"].dropna().unique()):
            conds.append((f"{n}-back {h}", f"h{n}", h))
    for rd in ("W1", "W2"):
        for which in ("pre", "post"):
            base = H[(H.reading == rd) & (H.which == which)]
            for name, col, val in conds:
                d = base if col is None else base[base[col] == val]
                for m, g in d.groupby("mouse"):
                    e, p = achda.t1(g[g.hemi == "TelC"].v, g[g.hemi == "mCherry"].v)
                    res.append(dict(reading=rd, window=which, condition=name, mouse=m, n=int((g.hemi == "TelC").sum()), diff=e, p=p))
    return pd.DataFrame(res)

def consistency(T, n_mice):
    g = T.dropna(subset=["p"]).groupby(["reading", "window", "condition"])
    c = g.apply(lambda x: pd.Series(dict(mice=len(x), sig_pos=int(((x.p < 0.05) & (x["diff"] > 0)).sum()), sig_neg=int(((x.p < 0.05) & (x["diff"] < 0)).sum()))), include_groups=False)
    c["consistent"] = (c.sig_pos > n_mice / 2) | (c.sig_neg > n_mice / 2)
    return c

for label, mice, ch in (("dLight1.1", GROUPS["vls_TelC_dlight"], "green"), ("rDAh", GROUPS["vls_TelC_rdah"], "red")):
    H = hemi_rows(mice, ch)
    T = hemi_tests(H)
    C = consistency(T, len(mice))
    print(f"\n{label}: {H.mouse.nunique()} mice; {C.shape[0]} comparisons (reading x window x condition); comparisons with a significant difference "
          f"of one sign in more than half of the mice: {int(C.consistent.sum())}")
    print(C[C.index.get_level_values("condition").isin(["all", "rewarded", "unrewarded"])].to_string())
    if C.consistent.any():
        print("consistent comparisons:"); print(C[C.consistent].to_string())
    strict = T[T.p < 0.05].groupby("mouse").size()
    print("strict reading, significant comparisons per mouse (of %d each):" % (T.groupby("mouse").size().iloc[0]), strict.to_dict())
    print("expected by chance at p < 0.05: about %.0f per mouse" % (0.05 * T.groupby("mouse").size().iloc[0]))

**Result: not reproduced.** For rDAh no comparison of the TelC and mCherry hemispheres (504, over windows, readings, trial
types and histories) is significant with one sign in more than half of the 4 mice. For dLight1.1, 34 of 592 are: for example,
after side entry in unrewarded trials DA is lower in the TelC hemisphere in 3 of 4 mice under both windows, and before side
entry over all trials higher in 3 of 4 under W1. By the strict count every mouse has significant comparisons (dLight1.1 54 to
224 of 596 per mouse, rDAh 1 to 58 of 488, against about 30 and 24 expected by chance). The comparison is between two fibres,
so placement differences enter it.

## ed8g-vls-telc-ach-suppressed and ed8f-amplitude-no-consistent-effect

> ed8g: In the simultaneous rDAh and Ach3.0 recordings, the TelC hemisphere has suppressed Ach release, validating the
> perturbation.

> ed8f: Loss of local Ach release does not consistently change the amplitude of DA sensor transients (dF/F0).

In [ ]:
# ed8g-vls-telc-ach-suppressed (Ach3.0, green channels, of the rDAh cohort) and ed8f-amplitude-no-consistent-effect
def dff_range(s, ch, reading, rew=True, span=2.0):
    """Peak-to-trough range of the trial-averaged dF/F0 within `span` s after side entry."""
    x = achda.dff(s["raw"][ch], reading)
    t = achda.photometry_trials(s)
    if rew is not None:
        t = t[t.was_rewarded == rew]
    idx = t.photometry_side_in_index.values[:, None] + np.arange(0, int(round(span / achda.DT)) + 1)[None, :]
    seg, _ = achda.segments(x, idx)
    m = seg.mean(0)
    return m.max() - m.min(), m.max(), len(seg)
rows = []
for m in GROUPS["vls_TelC_rdah"]:
    ts = TELC_SIDE[m]
    for s in [s for s in S if s["mouse"] == m]:
        for hemi, ch in (("TelC", "green" + ts), ("mCherry", "green" + ("R" if ts == "L" else "L"))):
            t = achda.photometry_trials(s)
            for rd in ("W1", "W2"):
                seg, _ = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, "post", rd))
                rows.append(pd.DataFrame({"mouse": m, "session": s["date"], "hemi": hemi, "reading": rd, "v": seg.max(1) - seg.min(1)}))
            for f0 in ("median", "rolling"):
                if ch in s["raw"]:
                    r, _, n = dff_range(s, ch, f0)
                    rows.append(pd.DataFrame({"mouse": [m], "session": [s["date"]], "hemi": [hemi], "reading": [f"dF/F0 {f0}"], "v": [r]}))
E8 = pd.concat(rows, ignore_index=True)
out = []
for (m, rd), g in E8.groupby(["mouse", "reading"]):
    a, b = g[g.hemi == "TelC"].v, g[g.hemi == "mCherry"].v
    e, p = achda.t1(a, b) if rd in ("W1", "W2") else (a.mean() - b.mean(), np.nan)
    out.append(dict(mouse=m, reading=rd, TelC=a.mean(), mCherry=b.mean(), diff=e, p_T1=p, n_TelC=len(a)))
print("ed8g, Ach in the TelC minus the mCherry hemisphere (W1/W2: delta-Ach per trial, T1; dF/F0: range of the rewarded-trial average within 2 s, mean over sessions):")
print(pd.DataFrame(out).round(4).to_string(index=False))
# ed8f: peak dF/F0 of the trial-averaged DA after side entry (rewarded trials, 2 s), TelC minus mCherry hemisphere
rows = []
for label, mice, col in (("dLight1.1", GROUPS["vls_TelC_dlight"], "green"), ("rDAh", GROUPS["vls_TelC_rdah"], "red")):
    for m in mice:
        ts = TELC_SIDE[m]
        for f0 in ("median", "rolling"):
            v = {"TelC": [], "mCherry": []}
            for s in [s for s in S if s["mouse"] == m]:
                for hemi, ch in (("TelC", col + ts), ("mCherry", col + ("R" if ts == "L" else "L"))):
                    if ch in s["raw"]:
                        v[hemi].append(dff_range(s, ch, f0)[1])
            rows.append(dict(sensor=label, mouse=m, F0=f0, sessions=len(v["TelC"]), peak_TelC=np.mean(v["TelC"]), peak_mCherry=np.mean(v["mCherry"]),
                             diff=np.mean(v["TelC"]) - np.mean(v["mCherry"])))
e8f = pd.DataFrame(rows)
print("\ned8f, peak dF/F0 of the rewarded-trial average within 2 s of side entry:"); print(e8f.round(4).to_string(index=False))
print(e8f.groupby(["sensor", "F0"])["diff"].apply(lambda d: f"signs: {int((d > 0).sum())} positive, {int((d < 0).sum())} negative").to_string())

**Result: reproduced, resting on an assumption (ed8g-vls-telc-ach-suppressed).** The dF/F0 range of the rewarded-trial Ach
average is lower in the TelC hemisphere in all 4 mice under both F0 readings (0.013 against 0.019, 0.005 against 0.021, 0.005
against 0.051 and 0.006 against 0.042 under the median F0). The z-scored delta-Ach (reading a) is lower with T1 p < 0.05 in
S1142 only under W1 and in three mice under W2, and higher in S952 under both. The outcome takes reading (b): z-scoring each
channel removes the loss of amplitude that TelC causes, as the claims file notes. In S1142 and S1143 the TelC hemisphere is
taken as the one carrying virus 222, which the files do not name.

**Result: blocked, methods underspecified (ed8f-amplitude-no-consistent-effect).** The paper does not say which DA sensor
Extended Data Fig. 8f shows. For dLight1.1 the peak dF/F0 is larger in the TelC hemisphere in all 4 mice (+0.008 to +0.030),
a consistent effect; for rDAh the sign is positive in 2 mice and negative in 2, under both F0 readings.

## ed7e-cin-activation-evokes-da

> Optogenetic activation of Chrimson-expressing VLS CINs during the task evokes a small, significant DA release, without
> changing decision or side-port occupancy times.

In [ ]:
# ed7e-cin-activation-evokes-da: Chrimson in the left VLS (redL, virus 216), dLight1.1 z-scored in the left hemisphere (greenL);
# stimulated trials = a laser burst starting at side entry (0 or 1 sample); peak z in the 1.5 s from side entry minus the mean
# of the 1.5 s before. The right-hemisphere dLight channel and all raw traces are absent from these files.
CHR = figmice("EDFig12")
rows = []
for s in [s for s in S if s["mouse"] in CHR]:
    L = s["events"]["laser_or_unused"]; starts = L[np.r_[True, np.diff(L) > 10]] if len(L) else L
    t = achda.photometry_trials(s); si = t.photometry_side_in_index.values
    stim = np.isin(si, starts) | np.isin(si + 1, starts)
    n15 = int(round(1.5 / achda.DT))
    z = s["z"]["greenL"]
    post, kp = achda.segments(z, si[:, None] + np.arange(0, n15 + 1)[None, :]); pre, kb = achda.segments(z, si[:, None] + np.arange(-n15, 0)[None, :])
    ok = kp & kb
    post, pre = z[si[ok][:, None] + np.arange(0, n15 + 1)[None, :]], z[si[ok][:, None] + np.arange(-n15, 0)[None, :]]
    change = post.max(1) - pre.mean(1); st = stim[ok]; tt = t[ok]
    occ = (tt.photometry_side_out_index - tt.photometry_side_in_index).values * achda.DT
    dec = (tt.photometry_side_in_index - tt.photometry_center_in_index).values * achda.DT
    rows.append(dict(mouse=s["mouse"], trials=int(ok.sum()), stim_trials=int(st.sum()), change_stim=change[st].mean(), change_nostim=change[~st].mean(),
                     p_T1=achda.t1(change[st], change[~st])[1], occ_stim=occ[st].mean(), occ_nostim=occ[~st].mean(), dec_stim=dec[st].mean(), dec_nostim=dec[~st].mean(),
                     rew_frac_stim=tt.was_rewarded.values[st].mean(), rew_frac_nostim=tt.was_rewarded.values[~st].mean()))
e7 = pd.DataFrame(rows)
print(e7.round(3).to_string(index=False))
for a, b, lab in (("change_stim", "change_nostim", "peak dLight change"), ("dec_stim", "dec_nostim", "decision time"), ("occ_stim", "occ_nostim", "side-port occupancy")):
    print(f"{lab}: stimulated {e7[a].mean():.3f}, unstimulated {e7[b].mean():.3f}, paired two-sided t-test across {len(e7)} mice p = {stats.ttest_rel(e7[a], e7[b]).pvalue:.4g}")
# rewarded and unrewarded trials separately (a variant)
rows = []
for s in [s for s in S if s["mouse"] in CHR]:
    L = s["events"]["laser_or_unused"]; starts = L[np.r_[True, np.diff(L) > 10]]
    t = achda.photometry_trials(s); si = t.photometry_side_in_index.values; stim = np.isin(si, starts) | np.isin(si + 1, starts)
    n15 = int(round(1.5 / achda.DT)); z = s["z"]["greenL"]
    ok = (si - n15 >= 0) & (si + n15 < len(z))
    ch = z[si[ok][:, None] + np.arange(0, n15 + 1)[None, :]].max(1) - z[si[ok][:, None] + np.arange(-n15, 0)[None, :]].mean(1)
    for rew in (True, False):
        r = t.was_rewarded.values[ok] == rew
        rows.append(dict(mouse=s["mouse"], rewarded=rew, stim=ch[stim[ok] & r].mean(), nostim=ch[~stim[ok] & r].mean()))
v = pd.DataFrame(rows)
for rew, g in v.groupby("rewarded"):
    print(f"{'rewarded' if rew else 'unrewarded'} trials: stimulated {g.stim.mean():.3f}, unstimulated {g.nostim.mean():.3f}, higher when stimulated in {int((g.stim > g.nostim).sum())} of {len(g)} mice, paired p = {stats.ttest_rel(g.stim, g.nostim).pvalue:.4g}")

**Result: not reproduced.** Stimulated trials (a laser burst starting at side entry, 45 to 72 per mouse) have a smaller peak
dLight1.1 change from the 1.5 s before entry than unstimulated trials (1.466 against 1.663 z, lower in 7 of 7 mice, paired
p = 5.6e-06); the same holds in rewarded and in unrewarded trials separately (variant above). Decision time (1.41 against 1.44 s,
p = 0.090) and side-port occupancy (2.10 against 2.10 s, p = 0.997) do not differ. Reading (a), Chrimson against mCherry hemisphere, cannot be run: the files hold no
trace of the right-hemisphere dLight1.1 channel, and no raw traces, so dF/F0 cannot be formed either.

## fig4g-cko-ach-suppression-lost and fig4g-cko-da-unchanged

> fig4g-ach: In Drd2-cKO mice the two dips in Ach that coincide with DA rises are lost, so Ach signals are significantly larger
> than in Drd2 f/f and wild-type mice.

> fig4g-da: DA dynamics during the task do not differ significantly across WT, Drd2 f/f and Drd2-cKO mice.

In [ ]:
# fig4g-cko-ach-suppression-lost and fig4g-cko-da-unchanged: windows of aa_processStats_final_Fig4_cKO_Ach_Words.m
# (post +1..+18 and +9..+16; all trials, pre -18..-1 and -18..-9 samples about side entry); Ach3.0 = greenL, rDAh = redL
WIN4 = {"post +1..+18": np.arange(1, 19), "post +9..+16": np.arange(9, 17), "pre -18..-1": np.arange(-18, 0), "pre -18..-9": np.arange(-18, -8)}
f4keys = set(zip(uses[uses.Figure == "Fig4"].Mouse, uses[uses.Figure == "Fig4"].date))
GENO = {m: g for g in ("wt", "ff", "cko") for m in GROUPS[g]}
rows = []
for s in S:
    if s["mouse"] not in GENO or (s["mouse"], s["date"]) not in f4keys:
        continue
    t = achda.photometry_trials(s)
    for sig, ch, meas in (("Ach", "greenL", "range"), ("DA", "redL", "mean"), ("Ach mean (variant)", "greenL", "mean")):
        if ch not in s["z"]:
            continue
        for wn, off in WIN4.items():
            seg, keep = achda.segments(s["z"][ch], t.photometry_side_in_index.values[:, None] + off[None, :])
            rows.append(pd.DataFrame({"mouse": s["mouse"], "geno": GENO[s["mouse"]], "signal": sig, "window": wn, "rew": t.was_rewarded.values[keep], "v": achda.measure(seg, meas)}))
G4 = pd.concat(rows, ignore_index=True)
print("mice per genotype with each signal:", G4.groupby(["signal", "geno"]).mouse.nunique().to_dict())
res = []
for (sig, wn), g in G4.groupby(["signal", "window"]):
    for tt, d in (("all", g), ("rewarded", g[g.rew]), ("unrewarded", g[~g.rew])):
        if wn.startswith("pre") and tt != "all":
            continue
        for unit in ("(a) mouse means", "(b) trials pooled"):
            dd = d.groupby(["geno", "mouse"]).v.mean().reset_index() if unit.startswith("(a)") else d
            v = {k: dd[dd.geno == k].v.values for k in ("wt", "ff", "cko")}
            res.append(dict(signal=sig, window=wn, trials=tt, unit=unit, mean_wt=v["wt"].mean(), mean_ff=v["ff"].mean(), mean_cko=v["cko"].mean(),
                            anova_p=stats.f_oneway(v["wt"], v["ff"], v["cko"]).pvalue, p_cko_vs_ff=stats.ttest_ind(v["cko"], v["ff"]).pvalue,
                            p_cko_vs_wt=stats.ttest_ind(v["cko"], v["wt"]).pvalue))
F4 = pd.DataFrame(res)
print(F4[F4.signal == "Ach"].round(5).to_string(index=False))
print(); print(F4[F4.signal == "DA"].round(5).to_string(index=False))
print("\nVariant, mean z-scored Ach:"); print(F4[F4.signal.str.startswith("Ach mean")].round(5).to_string(index=False))

**Result: not reproduced (fig4g-cko-ach-suppression-lost).** With one value per mouse (reading a) the ANOVA of delta-Ach
across genotypes is not significant in any window (p = 0.73 before side entry, 0.89 and 0.98 after it in rewarded trials).
With trials pooled (reading b) every ANOVA gives p < 1e-5, but Drd2-cKO delta-Ach is below WT before side entry (2.98 against
3.10) and in the rewarded +9 to +16 window (2.10 against 2.15). The mean z-scored Ach, a variant (the paper's code computes
both measures and the Extended Data Fig. 11d legend does not say which it plots), does separate the genotypes with one value per
mouse: before side entry (-18 to -1) ANOVA p = 0.0005, Drd2-cKO above Drd2 f/f (p = 0.003) and WT (p = 0.001); after side entry
in rewarded trials (+9 to +16) p = 0.0001, cKO above both (p = 0.003 and 0.00006).

**Result: not reproduced (fig4g-cko-da-unchanged).** Under reading (a) the ANOVA of mean rDAh is significant in the +9 to +16
window over all trials (p = 0.039; Drd2-cKO 0.39 against 0.26 and 0.22 z) and in rewarded trials (p = 0.049); the other
windows give p ≥ 0.11.

## fig5b-glutamate-not-lateralized, fig5c-glutamate-like-ach and fig5d-terminal-calcium-coincides

> fig5b: Striatal glutamate varies during the task and does not differ between ipsiversive and contraversive trials.

> fig5c: Glutamate is suppressed before the choice and rises at side-port entry like Ach, and in rewarded trials shows a
> sustained rise during consumption that is absent in unrewarded trials.

> fig5d: Calcium activity in cortical and thalamic terminals in the VLS coincides with glutamate release in rewarded and
> unrewarded trials.

In [ ]:
# Fig. 5 cohorts: glutamate (iGluSnFR, labelled "glutamate sensor"), S1299-S1302; Ach3.0 with jRCaMP1b (thalamic terminals), the
# Fig5 WT mice with a jRcamp1b channel; retrograde GCaMP8 (cortical terminals), WT74-WT79
GLU = figmice("Fig5", "Chat Jax")
JRC = sorted({s["mouse"] for s in S if s["mouse"] in figmice("Fig5", "WT C57") and any(s["meta"][c]["sensor"] == "jRcamp1b" for c in s["meta"])})
GC8 = sorted({s["mouse"] for s in S if any("Gcamp8" in s["meta"][c]["sensor"] for c in s["meta"])})
f5 = sessions_of(GLU + JRC + GC8, "Fig5")
def chan_with(s, test):
    return [c for c in achda.CHANNELS if c in s["z"] and test(s["meta"][c]["sensor"])]
print("glutamate mice", GLU, "; jRCaMP1b mice", JRC, "; GCaMP8 mice", GC8)
print("sessions with each stream:", {lab: sum(bool(chan_with(s, f)) for s in f5) for lab, f in (("glutamate", lambda x: x == "glutamate sensor"), ("Ach3.0", lambda x: x == "Ach3.0"), ("jRcamp1b", lambda x: x == "jRcamp1b"), ("GCaMP8", lambda x: "Gcamp8" in x))})
# fig5b-glutamate-not-lateralized
rows = []
for s in [s for s in f5 if s["mouse"] in GLU]:
    t = achda.photometry_trials(s)
    for ch in chan_with(s, lambda x: x == "glutamate sensor"):
        ipsi = np.where(s["meta"][ch]["hemisphere"] == "left", t.chose_left.values, ~t.chose_left.values)
        for rd in ("W1", "W2"):
            for which in ("pre", "post"):
                seg, keep = achda.segments(s["z"][ch], achda.window_idx(t.photometry_side_in_index.values, which, rd))
                rows.append(pd.DataFrame({"mouse": s["mouse"], "chan": ch, "reading": rd, "which": which, "ipsi": ipsi[keep], "v": seg.mean(1)}))
G5 = pd.concat(rows)
out = []
for (ch, rd, which, m), g in G5.groupby(["chan", "reading", "which", "mouse"]):
    e, p = achda.t1(g[g.ipsi].v, g[~g.ipsi].v); out.append(dict(chan=ch, reading=rd, window=which, mouse=m, ipsi_minus_contra=e, p=p))
out = pd.DataFrame(out)
print("\nfig5b, glutamate ipsilateral minus contralateral, T1 per mouse:"); print(out.pivot_table(index=["chan", "reading", "window"], columns="mouse", values=["ipsi_minus_contra", "p"]).round(4).to_string())
print(out.groupby(["chan", "reading", "window"]).p.apply(lambda p: int((p < 0.05).sum())).rename("mice with p < 0.05").to_string())
# trial-averaged traces about side entry (-2 to +3 s) and centre entry, per mouse
SPAN = np.arange(int(round(-2 / achda.DT)), int(round(3 / achda.DT)) + 1)
def mouse_trace(mice, test, rew=None, align="side"):
    out = {}
    for m in mice:
        segs = []
        for s in [s for s in f5 if s["mouse"] == m]:
            chs = chan_with(s, test)
            if not chs:
                continue
            t = achda.photometry_trials(s)
            if rew is not None:
                t = t[t.was_rewarded == rew]
            ev = (t.photometry_side_in_index if align == "side" else t.photometry_center_in_index).values
            seg, _ = achda.segments(s["z"][chs[0]], ev[:, None] + SPAN[None, :]); segs.append(seg)
        if segs:
            out[m] = np.vstack(segs).mean(0)
    return out
is_glu, is_ach, is_jrc, is_gc8 = (lambda x: x == "glutamate sensor"), (lambda x: x == "Ach3.0"), (lambda x: x == "jRcamp1b"), (lambda x: "Gcamp8" in x)
tsec = SPAN * achda.DT; z0 = int(np.flatnonzero(SPAN == 0)[0])
glu = mouse_trace(GLU, is_glu); glu_c = mouse_trace(GLU, is_glu, align="center"); ach = mouse_trace(JRC, is_ach)
gm, gcm, am = np.mean(list(glu.values()), 0), np.mean(list(glu_c.values()), 0), np.mean(list(ach.values()), 0)
pre_w1 = gm[z0 - 17:z0].mean()
sel = (tsec >= -1) & (tsec <= 1)
rise = gm[z0:z0 + 10].max() - gm[z0]
print(f"\nfig5c: {len(glu)} glutamate mice, {len(ach)} Ach mice. Mean glutamate in the pre window (W1) {pre_w1:.3f} against its value at centre entry {gcm[z0]:.3f}; "
      f"rise within 0.5 s of side entry (max minus value at entry) {rise:.3f}; r(glutamate, Ach) from -1 to +1 s = {np.corrcoef(gm[sel], am[sel])[0, 1]:.3f}")
rows = []
for m in GLU:
    v = {}
    for rew in (True, False):
        segs = []
        for s in [s for s in f5 if s["mouse"] == m]:
            t = achda.photometry_trials(s); t = t[t.was_rewarded == rew]
            seg, _ = achda.segments(s["z"][chan_with(s, is_glu)[0]], t.photometry_side_in_index.values[:, None] + np.arange(int(1 / achda.DT), int(3 / achda.DT) + 1)[None, :]); segs.append(seg.mean(1))
        v[rew] = np.concatenate(segs)
    e, p = achda.t1(v[True], v[False]); rows.append(dict(mouse=m, rewarded=v[True].mean(), unrewarded=v[False].mean(), diff=e, p=p))
print("glutamate 1-3 s after side entry, rewarded against unrewarded:"); print(pd.DataFrame(rows).round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3)); ax.plot(tsec, gm, label="glutamate"); ax.plot(tsec, am, label="Ach"); ax.axvline(0, ls=":"); ax.legend(); ax.set_xlabel("s from side entry"); plt.show()
print("plotted ranges: glutamate %.2f to %.2f, Ach %.2f to %.2f" % (gm.min(), gm.max(), am.min(), am.max()))
# fig5d-terminal-calcium-coincides
for rew in (True, False):
    g = np.mean(list(mouse_trace(GLU, is_glu, rew).values()), 0)
    for lab, mice, test in (("jRCaMP1b, thalamic", JRC, is_jrc), ("GCaMP8, cortical", GC8, is_gc8)):
        tr = mouse_trace(mice, test, rew)
        c = np.mean(list(tr.values()), 0)
        print(f"fig5d, {'rewarded' if rew else 'unrewarded'}, {lab}: {len(tr)} mice, r(mean calcium, mean glutamate) over -2..+3 s = {np.corrcoef(c, g)[0, 1]:.3f}; "
              f"per mouse r = {[round(np.corrcoef(v, g)[0, 1], 2) for v in tr.values()]}")

**Result: not reproduced (fig5b-glutamate-not-lateralized).** After side entry glutamate differs between ipsilateral and
contralateral trials in 4 of 4 mice on the left fibre (W1 and W2, +0.09 to +0.43 z) and 3 of 4 on the right; before side entry
2 or 3 of 4.

**Result: partially reproduced (fig5c-glutamate-like-ach).** Glutamate rises within 0.5 s of side entry (by 0.229 z), its mean
trace correlates with the Ach trace of the 9 jRCaMP1b mice from -1 to +1 s (r = 0.410), and from 1 to 3 s after side entry it
is higher in rewarded than unrewarded trials in 4 of 4 mice (by 0.53 to 0.83 z, p < 0.0001). Before side entry it is not
below its value at centre entry (-0.335 against -0.362 z).

**Result: reproduced (fig5d-terminal-calcium-coincides).** The mean jRCaMP1b trace (thalamic terminals, 9 mice where the paper
has 6) correlates with the mean glutamate trace at r = 0.849 (rewarded) and 0.676 (unrewarded), and the GCaMP8 trace (cortical
terminals, 5 mice) at 0.912 and 0.790.

## fig5ef-input-telc-dampens-ach and fig5ef-input-specific-ach

> fig5ef-dampens: Silencing cortical or thalamic glutamatergic inputs with TelC robustly lowers Ach sensor dF/F0 across all
> trials, consistently across mice.

> fig5ef-specific: In unrewarded trials, loss of cortical but not thalamic input perturbs Ach after side-port entry, and loss
> of thalamic input shifts the timing of Ach transients more than loss of cortical input.

In [ ]:
# fig5ef: Ach3.0 (greenL) in cortical (Vglut1-Cre) and thalamic (Vglut2-Cre) TelC and mCherry mice
INP = {"cortex": ("ctx_TelC", "ctx_mCh"), "thalamus": ("thal_TelC", "thal_mCh")}
rows = []
for inp, (gt, gm_) in INP.items():
    for grp in (gt, gm_):
        for m in GROUPS[grp]:
            for s in [s for s in S if s["mouse"] == m]:
                ch = "greenL"
                row = dict(input=inp, group="TelC" if grp == gt else "mCherry", mouse=m, has_z=ch in s["z"], has_raw=ch in s["raw"])
                t = achda.photometry_trials(s)
                if ch in s["raw"]:
                    for f0 in ("median", "rolling"):
                        row[f"range_dff_{f0}"] = dff_range(s, ch, f0, rew=None)[0]
                if ch in s["z"]:
                    tn = t[~t.was_rewarded]
                    for rd in ("W1", "W2"):
                        seg, _ = achda.segments(s["z"][ch], achda.window_idx(tn.photometry_side_in_index.values, "post", rd))
                        row[f"dAch_{rd}"] = (seg.max(1) - seg.min(1)).mean()
                    seg, _ = achda.segments(s["z"][ch], tn.photometry_side_in_index.values[:, None] + np.arange(0, int(2 / achda.DT) + 1)[None, :])
                    row["peak_latency_s"] = seg.mean(0).argmax() * achda.DT
                rows.append(row)
E5 = pd.DataFrame(rows)
print(E5.round(4).to_string(index=False))
for inp in INP:
    d = E5[E5.input == inp]
    for col in [c for c in ("range_dff_median", "range_dff_rolling") if c in d]:
        a, b = d[d.group == "TelC"][col].dropna(), d[d.group == "mCherry"][col].dropna()
        print(f"fig5ef dampens, {inp}, {col}: TelC {list(a.round(4))}, mCherry {list(b.round(4))}; every TelC below the smallest mCherry: "
              f"{bool(len(a) and len(b) and (a < b.min()).all()) if len(b) else 'no mCherry raw traces'}")
    for rd in ("W1", "W2"):
        a, b = d[d.group == "TelC"][f"dAch_{rd}"], d[d.group == "mCherry"][f"dAch_{rd}"]
        print(f"fig5ef specific, {inp}, unrewarded delta-Ach {rd}: TelC {a.mean():.3f}, mCherry {b.mean():.3f}, two-sample t-test p = {stats.ttest_ind(a, b).pvalue:.4f}")
    lat = d.groupby("group").peak_latency_s.mean()
    print(f"  {inp}: mean latency of the unrewarded Ach peak, TelC {lat['TelC']:.3f} s, mCherry {lat['mCherry']:.3f} s, shift {abs(lat['TelC'] - lat['mCherry']):.3f} s")

**Result: blocked, data missing (fig5ef-input-telc-dampens-ach).** For the cortex the dF/F0 range of every TelC mouse (0.011 to
0.016) is below the smallest mCherry range (0.022) under both F0 readings. For the thalamus the three mCherry mice have no raw
trace in the files, so their dF/F0 cannot be formed.

**Result: not reproduced (fig5ef-input-specific-ach).** Unrewarded delta-Ach does not differ between cortical TelC and mCherry
mice (p = 0.40 under W1, 0.26 under W2), and the peak latency shifts less for thalamic (0.108 s) than for cortical TelC
(0.162 s).

## ed2op-mutant-sensors-flat

> The DA and Ach transients depend on neuromodulator binding, being absent with the ligand-binding-site mutant sensors.

In [ ]:
# ed2op-mutant-sensors-flat: Ach3.0 and its ligand-binding mutant (EDFig3 mice); no dLight1.1 mutant is in the dataset
rows = []
for s in [s for s in S if s["mouse"] in figmice("EDFig3")]:
    for ch in s["z"]:
        sens = s["meta"][ch]["sensor"]
        row = dict(mouse=s["mouse"], date=s["date"], chan=ch, sensor=sens, has_raw=ch in s["raw"])
        if ch in s["raw"]:
            for f0 in ("median", "rolling"):
                row[f"range_{f0}"] = dff_range(s, ch, f0, rew=True)[0]
        t = achda.photometry_trials(s); t = t[t.was_rewarded]
        seg, _ = achda.segments(s["z"][ch], t.photometry_side_in_index.values[:, None] + np.arange(0, int(2 / achda.DT) + 1)[None, :])
        row["range_z"] = np.ptp(seg.mean(0))
        rows.append(row)
mut = pd.DataFrame(rows)
print(mut.round(4).to_string(index=False))
print("sensors anywhere in the dataset with 'mutant' in the name:", sorted({s["meta"][c]["sensor"] for s in S for c in s["meta"] if "mutant" in s["meta"][c]["sensor"]}))

**Result: not reproduced.** In WT14 the mutant's dF/F0 range is 0.15 of the Ach3.0 range in the other hemisphere (0.0042
against 0.0280), below a quarter; in WT15 it is 0.33 (0.0086 against 0.0257). WT12 carries the mutant in both hemispheres. No
dLight1.1 mutant is in the dataset, so the DA part cannot be tested.

## GLMs: ed5-base-glm-r2, ed5f-leave-out and ed5-history-glm-r2

> ed5-base: A base GLM of behavioural events explains about a fifth of the variance of trial-associated DA and Ach signals
> (test R2 0.206 for each).

> ed5f: Leaving out side entry or reward raises the GLM's MSE for both DA and Ach, leaving out side exit or lick raises it for
> Ach, and leaving out centre entry or centre exit alone does not change it.

> ed5-history: Adding side entries split by the eight action-outcome histories improves the GLM, to a test R2 of 0.213 for DA
> and 0.214 for Ach, with a lower MSE and no overfitting.

The cohort is the export table's EDFig10 sessions; the GLM code is not published, so the predictor set (P1, P2), the pooling
(G1 to G3) and Y are run as the claims file names them.

In [ ]:
# GLMs (Methods, GLM): OLS on trial segments, 41 shifts per predictor, 50/50 trial split per run, test R2 averaged over runs.
# Pooling readings: G1 one model over all sessions of all mice, G2 one per mouse, G3 one per session (R2 averaged).
P1_NAMES = ["centre_entry", "centre_exit", "side_entry", "side_exit", "lick", "reward"]
def glm_units(cohort, predictor_set, history, continuous=None):
    units = []
    for s, ch_y, mouse in cohort:
        cont = {k: s["z"][c].astype(float) for k, c in (continuous(s) if continuous else {}).items()}
        X, y, trial_of, names = achda.glm_session(s, ch_y, achda.glm_events(s, predictor_set, history), cont)
        units.append(dict(mouse=mouse, session=s["date"], X=X, y=y, trial=trial_of, names=names))
    return units

def run_glm(units, runs, rng, leave_out=None, folds=0):
    names = units[0]["names"]
    res = {g: [] for g in ("G1", "G2", "G3")}; kernels = []; val = []
    for r in range(runs):
        tr_st, te_st, fold_st = [], [], []
        for u in units:
            trials = np.unique(u["trial"]); test = rng.choice(trials, len(trials) // 2, replace=False)
            te = np.isin(u["trial"], test)
            tr_st.append(achda.suff_stats(u["X"], u["y"], ~te)); te_st.append(achda.suff_stats(u["X"], u["y"], te))
            if folds:
                train_trials = np.setdiff1d(trials, test); fs = []
                for f in range(folds):
                    v = rng.choice(train_trials, int(round(0.2 * len(train_trials))), replace=False)
                    fs.append(achda.suff_stats(u["X"], u["y"], np.isin(u["trial"], v)))
                fold_st.append(fs)
        groups = {"G1": [list(range(len(units)))],
                  "G2": [[i for i, u in enumerate(units) if u["mouse"] == m] for m in sorted({u["mouse"] for u in units})],
                  "G3": [[i] for i in range(len(units))]}
        for g, parts in groups.items():
            vals = []
            for idx in parts:
                trs = tr_st[idx[0]]; tes = te_st[idx[0]]
                for i in idx[1:]:
                    trs = achda.add_stats(trs, tr_st[i]); tes = achda.add_stats(tes, te_st[i])
                b = achda.fit_ols(trs)
                vals.append(achda.r2(tes, b))
                if g == "G1":
                    kernels.append(b)
                if folds:
                    models = {"full": None, **{f"-{p}": [j for j, n in enumerate(names) if not n.startswith(p + ":")] for p in leave_out}}
                    for mname, cols in models.items():
                        tot_sse, tot_n = 0.0, 0
                        for f in range(folds):
                            vs = fold_st[idx[0]][f]
                            for i in idx[1:]:
                                vs = achda.add_stats(vs, fold_st[i][f])
                            bf = achda.fit_ols(achda.add_stats(trs, vs, -1), cols)
                            tot_sse += achda.sse(vs, bf, cols); tot_n += vs["n"]
                        val.append(dict(run=r, reading=g, unit=len(vals) - 1, model=mname, val_mse=tot_sse / tot_n))
            res[g].append(float(np.mean(vals)))
    return {g: np.array(v) for g, v in res.items()}, np.array(kernels), names, pd.DataFrame(val)

def glm_table(results, label):
    rows = []
    for (sig, pset), (r2s, _, _, _) in results.items():
        for g, v in r2s.items():
            rows.append(dict(model=label, signal=sig, predictors=pset, pooling=g, R2_Y10=v.mean(), R2_Y3=v[:3].mean(), R2_min=v.min(), R2_max=v.max()))
    return pd.DataFrame(rows)

# Extended Data Fig. 5 cohort: the CSV's EDFig10 sessions (the GLM figure of the export table; the published figure is Extended Data Fig. 5).
# DA: dLight1.1 where the mouse has it (greenR, else greenL), otherwise rDAh (redL); Ach: Ach3.0 (greenL).
e5 = sessions_of(figmice("EDFig10"), "EDFig10")
def da_chan(s):
    for c in ("greenR", "greenL"):
        if c in s["z"] and s["meta"][c]["sensor"] == "dlight1.1":
            return c
    return "redL" if "redL" in s["z"] and s["meta"]["redL"]["sensor"] == "rDAh" else None
COH5 = {"DA": [(s, da_chan(s), s["mouse"]) for s in e5 if da_chan(s)],
        "Ach": [(s, "greenL", s["mouse"]) for s in e5 if "greenL" in s["z"] and s["meta"]["greenL"]["sensor"] == "Ach3.0"]}
for k, v in COH5.items():
    print(f"{k}: {len({m for _, _, m in v})} mice, {len(v)} sessions; channels {pd.Series([c for _, c, _ in v]).value_counts().to_dict()}")

In [ ]:
t0 = time.time()
rng = np.random.default_rng(131)
BASE, HIST = {}, {}
for sig in ("DA", "Ach"):
    for pset in ("P1", "P2"):
        BASE[(sig, pset)] = run_glm(glm_units(COH5[sig], pset, False), 10, rng, leave_out=P1_NAMES, folds=10)
        HIST[(sig, pset)] = run_glm(glm_units(COH5[sig], pset, True), 10, rng)
print(f"GLMs fitted in {time.time() - t0:.0f} s")
g5 = pd.concat([glm_table(BASE, "base"), glm_table(HIST, "history")])
print(g5.round(4).to_string(index=False))

In [ ]:
# ed5f-leave-out: validation MSE of the base model with each predictor left out (median and range over the 10 runs)
rows = []
for (sig, pset), (_, _, _, val) in BASE.items():
    v = val.groupby(["reading", "run", "model"]).val_mse.mean().reset_index()     # G2/G3: mean over units
    for g, d in v.groupby("reading"):
        full = d[d.model == "full"].val_mse
        for mname, dm in d.groupby("model"):
            rows.append(dict(signal=sig, predictors=pset, pooling=g, model=mname, median=dm.val_mse.median(), lo=dm.val_mse.min(), hi=dm.val_mse.max(),
                             rise_vs_full=dm.val_mse.median() - full.median(), above_full_range=dm.val_mse.min() > full.max(),
                             within_full_range=full.min() <= dm.val_mse.median() <= full.max()))
lo = pd.DataFrame(rows)
print(lo.round(5).to_string(index=False))
want_up = {"DA": ["-side_entry", "-reward"], "Ach": ["-side_entry", "-reward", "-side_exit", "-lick"]}
summ = []
for (sig, pset, g), d in lo.groupby(["signal", "predictors", "pooling"]):
    d = d.set_index("model")
    summ.append(dict(signal=sig, predictors=pset, pooling=g, rises_hold=bool(d.loc[want_up[sig], "above_full_range"].all()),
                     centre_unchanged=bool(d.loc[["-centre_entry", "-centre_exit"], "within_full_range"].all())))
print(pd.DataFrame(summ).to_string(index=False))
# ed5-history-glm-r2: test MSE of the history model below the base model's in every run

**Result: blocked, methods underspecified (ed5-base-glm-r2).** The tolerance is 0.185 to 0.227. One model per mouse (G2) puts
both R2 inside it (DA 0.203 with P1 and 0.212 with P2; Ach 0.215 and 0.226), and one per session (G3) puts DA inside (0.211,
0.216) but Ach above (0.243, 0.249); one model over all sessions (G1) gives 0.132 to 0.138 for DA and 0.159 to 0.166 for Ach.
Y = 3 and Y = 10 agree to 0.001. The readings disagree, which the criterion says blocks the claim.

**Result: blocked, methods underspecified (ed5f-leave-out).** It depends on ed5-base-glm-r2. As run, leaving out reward raises
the DA validation MSE beyond the full model's range under every reading, but leaving out side entry does not (rise 0.002 to
0.008); for Ach the four predictors raise it beyond the range under G1 with P1 and G3 with P2 only; leaving out centre entry or
centre exit keeps the median within the full model's range under every reading.

**Result: blocked, methods underspecified (ed5-history-glm-r2).** It depends on ed5-base-glm-r2. As run, the history model's
R2 is within tolerance under G2 and G3 for DA (0.214 to 0.223) and under G2 with P1 for Ach (0.232), and above or below it
otherwise (DA G1 0.145 and 0.150; Ach G1 0.181 and 0.187, G2 with P2 0.240, G3 0.252 and 0.257).

## Blocked claims

- fig2h-photometry-glm and ed11-cko-glm-r2 depend on ed5-history-glm-r2, which is blocked; they are blocked for the same
  reason and were not run.
- fig3d-telc-abolishes-ach: its 3 mice with Ach3.0 and TelC or mCherry in VLS CINs are not in the dataset. The Fig. 3
  sessions the export table lists all carry dLight1.1; the mice it lists for Fig. 3 that the dataset lacks (S731, S734, S803,
  S804, S998, S999, S1001) may be them. The within-mouse comparison of ed8g is a different design.
- fig3c-telc-blocks-cin-evoked-da-in-vitro and fig4ef-d2r-needed-for-cin-pause: slice amperometry and cell-attached
  recordings; the dataset holds no slice data.
- fig5a-inputs-to-vls: histology; not in the dataset.
- fig3g-dan-inhibition-lowers-da: no stGtACR2 session is in the dataset (no session has laser onsets outside the
  CIN-stimulation cohort); the six Fig4 "Chat Dat" mice the export table lists (S1183 to S1241) have no session in it.
- fig4bc-da-bidirectionally-controls-ach: the head-fixed sessions on the wheel are not in the dataset.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from
the paper's method.

| Claim | Outcome |
|---|---|
| link-manuscript-sessions | reproduced |
| ed1b-switch-after-block-transition | reproduced |
| ed1hj-rflr-predicts-switching | not reproduced |
| fig1e-reward-outcome-post | reproduced |
| fig1e-reward-outcome-not-pre | reproduced (assumption: window W1) |
| fig1e-lda-accuracy | reproduced |
| ed3-not-lateralized | not reproduced |
| fig1f-cue-omission | reproduced |
| fig1g-history-stay | reproduced |
| fig1h-switch-no-history-post | reproduced |
| fig1h-switch-da-approach | reproduced |
| ed4bc-iti-port-entries | not reproduced |
| ed5-base-glm-r2, ed5f-leave-out, ed5-history-glm-r2 | blocked: methods underspecified |
| fig2a-rdah-smaller-than-dlight | partially reproduced |
| ed6c-da-bilateral | not reproduced |
| fig2d-anticorrelated-lag | reproduced |
| fig2e-trial-and-noise-correlation | reproduced |
| fig2fg-off-diagonal | partially reproduced |
| fig2h-photometry-glm | blocked: methods underspecified |
| ed6b-separate-hemispheres | reproduced |
| fig3d-telc-abolishes-ach | blocked: data missing |
| fig3c-telc-blocks-cin-evoked-da-in-vitro | blocked: data missing |
| fig3e-striatum-wide-telc | reproduced |
| fig3f-vls-telc-da-unchanged | not reproduced |
| ed8g-vls-telc-ach-suppressed | reproduced (assumption: dF/F0 reading) |
| ed9jm-vls-telc-behaviour-normal | not reproduced |
| ed8f-amplitude-no-consistent-effect | blocked: methods underspecified |
| fig3g-dan-inhibition-lowers-da | blocked: data missing |
| ed7e-cin-activation-evokes-da | not reproduced |
| fig4bc-da-bidirectionally-controls-ach | blocked: data missing |
| fig4ef-d2r-needed-for-cin-pause | blocked: data missing |
| fig4g-cko-ach-suppression-lost | not reproduced |
| fig4g-cko-da-unchanged | not reproduced |
| ed11e-cko-anticorrelation-reduced | reproduced |
| ed11-cko-glm-r2 | blocked: methods underspecified |
| ed11fh-cko-general-behaviour-comparable | not reproduced |
| ed11i-cko-rflr | not reproduced |
| ed11j-cko-switching-impaired | not reproduced (assumption: sessions resampled) |
| fig5a-inputs-to-vls | blocked: data missing |
| fig5b-glutamate-not-lateralized | not reproduced |
| fig5c-glutamate-like-ach | partially reproduced |
| fig5d-terminal-calcium-coincides | reproduced |
| fig5ef-input-telc-dampens-ach | blocked: data missing |
| fig5ef-input-specific-ach | not reproduced |
| ed13-input-telc-behaviour | partially reproduced |
| ed2op-mutant-sensors-flat | not reproduced |

How the analysis differs from the paper's. Cohorts come from the export table, not from the paper's data folders, and
some differ in size from the paper's (Fig. 1: 14 DA and 16 Ach mice against 13 and 14; jRCaMP1b: 9 against 6; cortical TelC: 3
against 5). The behaviour claims of Extended Data Fig. 1 use the Fig. 1 mice. The LDA uses scikit-learn's shrinkage LDA in place
of MATLAB's optimised fitcdiscr. The RFLR is fitted by maximum likelihood with Nelder-Mead in place of the unpublished SGD
module. Decision time and ITI are taken from the trials table's entry and exit times. The whole-session cross-correlation uses
54-ms samples in place of 18-ms ones. The CIN-stimulation analysis uses z-scored traces of one hemisphere. Where the paper or
the claims file leaves a choice open, every reading named is shown, and the outcomes resting on one are marked.

What was found about the dataset. The files hold no figure, cohort or genotype labels and placeholder subject metadata
(age P0D, sex U), so most claims need the export repository's table, which the dataset's metadata links. The trials tables'
times are not on the photometry clock (offsets of -125 to +1.2 s), though their sample indices are. The trials tables' laser
columns are False where the laser channel shows stimulation at side entry. 193 named channels lack their raw trace and 30 also
their z-scored trace, which removes the right-hemisphere dLight1.1 of the CIN-stimulation mice and the raw traces of the
thalamic mCherry mice. The stored `baseline_f0` is not on the scale of the raw trace. The `sensor` field holds virus codes in
four mice. Nothing marks the cue-omission sessions. The slice, histology, head-fixed and DAN-inhibition data, the dLight1.1
mutant and 21 mice named in the export table are not in the dataset.